# 패키지 설치 및 데이터 로딩

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# 한글 폰트 설정 (Mac용)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 엑셀 읽기
df = pd.read_excel(os.path.join('Online Retail.xlsx'))

In [2]:
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


# 결측값 처리

In [3]:
def handle_missing_values(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """
    Online Retail 데이터셋의 결측값을 처리한다.
 
    Parameters
    ----------
    df : pd.DataFrame
        원본 데이터프레임. 최소 컬럼: InvoiceNo, StockCode, Description,
        Quantity, InvoiceDate, UnitPrice, CustomerID, Country.
    verbose : bool, default True
        처리 과정 로그 출력 여부.
 
    Returns
    -------
    pd.DataFrame
        결측값이 처리된 데이터프레임 (복사본).
    """
    df = df.copy()
    n_before = len(df)
 
    if verbose:
        print("=" * 60)
        print("결측값 처리 시작")
        print(f"  원본 데이터: {n_before:,}행")
        print(f"  CustomerID 결측: {df['CustomerID'].isnull().sum():,}행 "
              f"({df['CustomerID'].isnull().mean():.1%})")
        print(f"  Description 결측: {df['Description'].isnull().sum():,}행 "
              f"({df['Description'].isnull().mean():.1%})")
        print("=" * 60)
 
    # ── 1. CustomerID 결측 처리 ──────────────────────────────────
    # InvoiceNo에서 숫자 부분을 추출하여 대리 ID를 부여한다.
    # 기존 CustomerID 범위(최대 ~18,000대)와 겹치지 않도록 900,000을 더한다.
    invoice_nums = (
        df['InvoiceNo']
        .astype(str)
        .str.extract(r'(\d+)', expand=False)
        .astype(float)
    )
    mask_null_cid = df['CustomerID'].isnull()
    df.loc[mask_null_cid, 'CustomerID'] = invoice_nums[mask_null_cid] + 900_000
    df['CustomerID'] = df['CustomerID'].astype(int)
 
    if verbose:
        print(f"\n[1/2] CustomerID 처리 완료")
        print(f"  → 대리 ID 부여: {mask_null_cid.sum():,}행 (offset=900,000)")
        print(f"  → 남은 결측: {df['CustomerID'].isnull().sum()}")
 
    # ── 2. Description 결측 처리 ─────────────────────────────────
    # 2-1) 동일 StockCode의 첫 번째 유효 Description으로 매핑
    stock_desc_map = (
        df.dropna(subset=['Description'])
        .groupby('StockCode')['Description']
        .first()
    )
    n_desc_before = df['Description'].isnull().sum()
    df['Description'] = df['Description'].fillna(
        df['StockCode'].map(stock_desc_map)
    )
    n_filled = n_desc_before - df['Description'].isnull().sum()
 
    # 2-2) 매핑으로도 복구 불가한 행 삭제 (주로 환불건·단가 0원)
    n_remaining = df['Description'].isnull().sum()
    df = df.dropna(subset=['Description'])
 
    if verbose:
        print(f"\n[2/2] Description 처리 완료")
        print(f"  → StockCode 매핑으로 복구: {n_filled:,}행")
        print(f"  → 복구 불가 삭제: {n_remaining:,}행")
        print(f"\n최종 데이터: {len(df):,}행 "
              f"(삭제율: {1 - len(df)/n_before:.2%})")
        print("=" * 60)
 
    return df.reset_index(drop=True)

In [4]:
df_clean = handle_missing_values(df)
print(f"\n처리 후 결측값 확인:")
print(df_clean.isnull().sum())

결측값 처리 시작
  원본 데이터: 541,909행
  CustomerID 결측: 135,080행 (24.9%)
  Description 결측: 1,454행 (0.3%)

[1/2] CustomerID 처리 완료
  → 대리 ID 부여: 135,080행 (offset=900,000)
  → 남은 결측: 0

[2/2] Description 처리 완료
  → StockCode 매핑으로 복구: 1,342행
  → 복구 불가 삭제: 112행

최종 데이터: 541,797행 (삭제율: 0.02%)

처리 후 결측값 확인:
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


# 이상치 처리

In [5]:
import pandas as pd
import numpy as np
 
 
def handle_outliers(
    df: pd.DataFrame,
    winsorize_lower: float = 0.01,
    winsorize_upper: float = 0.99,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Online Retail 데이터셋의 이상치를 처리한다.
 
    Parameters
    ----------
    df : pd.DataFrame
        결측값 처리가 완료된 데이터프레임.
    winsorize_lower : float, default 0.01
        UnitPrice 윈저라이징 하한 백분위.
    winsorize_upper : float, default 0.99
        UnitPrice 윈저라이징 상한 백분위.
    verbose : bool, default True
        처리 과정 로그 출력 여부.
 
    Returns
    -------
    pd.DataFrame
        이상치가 처리된 데이터프레임 (복사본).
    """
    df = df.copy()
    n_before = len(df)
 
    if verbose:
        print("=" * 60)
        print("이상치 처리 시작")
        print(f"  입력 데이터: {n_before:,}행")
        print("=" * 60)
 
    # ── 1. 노이즈 제거: Quantity < 0 & UnitPrice == 0 ────────────
    # 수량이 음수이면서 단가가 0원인 행은 매출 기여가 없는
    # 시스템 로그성 데이터로, 분석 가치가 없다.
    noise_mask = (df['Quantity'] < 0) & (df['UnitPrice'] == 0)
    n_noise = noise_mask.sum()
    df = df[~noise_mask]
 
    if verbose:
        print(f"\n[1/3] 노이즈 제거 (Quantity < 0 & UnitPrice == 0)")
        print(f"  → 제거: {n_noise:,}행")
 
    # ── 2. 중복행 제거 ───────────────────────────────────────────
    # 완전히 동일한 행은 시스템 이중 기록일 가능성이 높다.
    # (선행연구: "deduplicate exact duplicates")
    dedup_cols = [
        'InvoiceNo', 'StockCode', 'Description',
        'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country',
    ]
    # 실제 존재하는 컬럼만 사용
    dedup_cols = [c for c in dedup_cols if c in df.columns]
    n_before_dedup = len(df)
    df = df.drop_duplicates(subset=dedup_cols, keep='first')
    n_duplicates = n_before_dedup - len(df)
 
    if verbose:
        print(f"\n[2/3] 중복행 제거 (exact duplicates)")
        print(f"  → 제거: {n_duplicates:,}행")
 
    # ── 3. UnitPrice 윈저라이징 ──────────────────────────────────
    # 극단적 단가(수만 파운드 등)는 Monetary 계산과 CLV 예측을 왜곡한다.
    # 행을 삭제하지 않고, 경계값으로 대체하여 데이터를 보존한다.
    # (선행연구: "winsorize unit prices at the 1st/99th percentiles")
    #
    # 윈저라이징은 UnitPrice > 0인 정상 거래 기준으로 경계값을 산출한 뒤
    # 전체 데이터에 적용한다. (단가 0원·음수는 별도 처리 대상이 아님)
    positive_prices = df.loc[df['UnitPrice'] > 0, 'UnitPrice']
    p_low = positive_prices.quantile(winsorize_lower)
    p_high = positive_prices.quantile(winsorize_upper)
 
    n_clipped_low = (df['UnitPrice'] < p_low).sum()
    n_clipped_high = (df['UnitPrice'] > p_high).sum()
 
    df['UnitPrice'] = df['UnitPrice'].clip(lower=p_low, upper=p_high)
 
    if verbose:
        print(f"\n[3/3] UnitPrice 윈저라이징 "
              f"({winsorize_lower:.0%} / {winsorize_upper:.0%})")
        print(f"  → 하한 경계: £{p_low:.2f} (적용: {n_clipped_low:,}행)")
        print(f"  → 상한 경계: £{p_high:.2f} (적용: {n_clipped_high:,}행)")
 
    # ── 요약 ─────────────────────────────────────────────────────
    n_after = len(df)
    n_refunds = df['InvoiceNo'].astype(str).str.startswith('C').sum()
 
    if verbose:
        print(f"\n{'─' * 60}")
        print(f"최종 데이터: {n_after:,}행 "
              f"(삭제율: {1 - n_after / n_before:.2%})")
        print(f"보존된 반품(C) 거래: {n_refunds:,}행 "
              f"(반품률 피처 활용 가능)")
        print("=" * 60)
 
    return df.reset_index(drop=True)

In [6]:
df_final = handle_outliers(df_clean)
print(f"\n최종 데이터 요약:")
print(df_final.describe())

이상치 처리 시작
  입력 데이터: 541,797행

[1/3] 노이즈 제거 (Quantity < 0 & UnitPrice == 0)
  → 제거: 1,239행

[2/3] 중복행 제거 (exact duplicates)
  → 제거: 5,268행

[3/3] UnitPrice 윈저라이징 (1% / 99%)
  → 하한 경계: £0.29 (적용: 5,415행)
  → 상한 경계: £18.00 (적용: 4,789행)

────────────────────────────────────────────────────────────
최종 데이터: 535,290행 (삭제율: 1.20%)
보존된 반품(C) 거래: 9,251행 (반품률 피처 활용 가능)

최종 데이터 요약:
            Quantity                    InvoiceDate      UnitPrice  \
count  535290.000000                         535290  535290.000000   
mean       10.030746  2011-07-04 10:11:42.592388608       3.266198   
min    -80995.000000            2010-12-01 08:26:00       0.290000   
25%         1.000000            2011-03-28 10:15:00       1.250000   
50%         3.000000            2011-07-19 15:13:00       2.080000   
75%        10.000000            2011-10-18 17:05:00       4.130000   
max     80995.000000            2011-12-09 12:50:00      18.000000   
std       217.274812                            NaN       3.306071 

# 파생변수 생성 

## 카테고리 1: 기본 마케팅 변수 (RFM 계열) 

In [7]:
def build_category1_rfm(df_final, reference_date='2011-12-10', verbose=True):
    """
    df_final(전처리 완료 데이터)로부터 카테고리 1 RFM 계열 7개 변수를 생성한다.

    Parameters
    ----------
    df_final : pd.DataFrame
        handle_missing_values() → handle_outliers() 완료된 데이터.
        필수 컬럼: CustomerID, InvoiceNo, InvoiceDate, Quantity, UnitPrice
    reference_date : str
        기준일 (기본값: '2011-12-10')
    verbose : bool
        True이면 생성 과정·기술통계·넛지 매핑을 출력

    Returns
    -------
    rfm : pd.DataFrame
        고객별 7개 RFM 파생변수 (CustomerID 포함)
    nudge_mapping : dict
        {변수명: {넛지_유형, 이론적_근거, 활용_목적, 선행연구_대응}}
    """

    ref_date = pd.to_datetime(reference_date)

    # ── Step 1: Revenue 컬럼 생성 ────────────────────────────────
    # df_final 원본을 변경하지 않기 위해 copy
    data = df_final.copy()
    data['Revenue'] = data['Quantity'] * data['UnitPrice']

    # ── Step 2: 정상 주문만 필터링 (RFM 계산용) ──────────────────
    #   - 선행연구: "remove administrative cancellations;
    #     negative quantities are treated as returns and netted"
    #   - 카테고리 1은 정상 거래 기반으로 계산
    #   - 반품·취소 데이터는 카테고리 6(긍정성 넛지)에서 별도 활용
    normal = data[
        (~data['InvoiceNo'].astype(str).str.startswith('C')) &
        (data['Quantity'] > 0) &
        (data['Revenue'] > 0)
    ].copy()

    if verbose:
        print("=" * 65)
        print("카테고리 1: 기본 마케팅 변수 (RFM 계열) 생성 시작")
        print("=" * 65)
        print(f"  전체 데이터: {len(data):,}행")
        print(f"  정상 주문만 필터링: {len(normal):,}행 "
              f"(제외: {len(data) - len(normal):,}행 — 반품·취소)")

    # ── Step 3: 주문(Invoice) 단위 중간 집계 ─────────────────────
    #   Avg_Items_per_Order를 정확히 계산하기 위해
    #   "주문별 수량 합계"를 먼저 구한 뒤 고객별 평균을 냄
    order_level = normal.groupby(
        ['CustomerID', 'InvoiceNo']
    ).agg(
        order_qty    = ('Quantity', 'sum'),
        order_rev    = ('Revenue',  'sum'),
        order_date   = ('InvoiceDate', 'max')
    ).reset_index()

    # ── Step 4: 고객 단위 집계 → 7개 변수 ────────────────────────
    cust = order_level.groupby('CustomerID')

    # 4-1. Recency: 마지막 구매일 ~ 기준일 일수
    recency = (ref_date - cust['order_date'].max()).dt.days
    recency.name = 'Recency'

    # 4-2. Frequency: 고유 InvoiceNo 수
    frequency = cust['InvoiceNo'].nunique()
    frequency.name = 'Frequency'

    # 4-3. Monetary: 총 Revenue
    monetary = cust['order_rev'].sum()
    monetary.name = 'Monetary'

    # 4-4. Active Months: 구매가 발생한 고유 월 수
    order_level['order_month'] = order_level['order_date'].dt.to_period('M')
    active_months = order_level.groupby('CustomerID')['order_month'].nunique()
    active_months.name = 'Active_Months'

    # 4-5. Purchase Span: 첫 구매 ~ 마지막 구매 일수
    purchase_span = cust['order_date'].agg(lambda x: (x.max() - x.min()).days)
    purchase_span.name = 'Purchase_Span'

    # 4-6. Avg Items per Order: 주문별 수량의 평균
    avg_items = cust['order_qty'].mean()
    avg_items.name = 'Avg_Items_per_Order'

    # ── Step 5: 결합 + AOV 계산 ──────────────────────────────────
    rfm = pd.concat(
        [recency, frequency, monetary, active_months,
         purchase_span, avg_items],
        axis=1
    )

    # 4-7. AOV: Monetary / Frequency
    rfm['AOV'] = rfm['Monetary'] / rfm['Frequency']

    # 컬럼 순서 정리 (제안서 순서)
    rfm = rfm[[
        'Recency', 'Frequency', 'Monetary', 'AOV',
        'Avg_Items_per_Order', 'Active_Months', 'Purchase_Span'
    ]]
    rfm = rfm.reset_index()

    # ── Step 6: 넛지 유형 매핑 메타데이터 ────────────────────────
    nudge_mapping = {
        'Recency': {
            '넛지_유형': '타성(inertia)',
            '이론적_근거': '현상유지 편향(status quo bias) — '
                         '구매 디폴트 상태에서의 이탈 정도',
            '활용_목적': 'Main 종속변수(이탈) 라벨링 기준',
            '선행연구_대응': 'Yang Liu(2025) recency와 동일 계산, '
                          '넛지 해석 추가'
        },
        'Frequency': {
            '넛지_유형': '일관성(consistency)',
            '이론적_근거': '인지부조화 이론 — 이전 행동과의 일관성 유지 동기',
            '활용_목적': '구매 충성도 측정',
            '선행연구_대응': 'Yang Liu(2025) frequency와 동일, '
                          '단순측정 효과 해석 추가'
        },
        'Monetary': {
            '넛지_유형': '비교성(comparability)',
            '이론적_근거': '사회비교 이론 — 고객 간 지출 비교의 앵커',
            '활용_목적': '고객 가치 측정, 카테고리 7 Country Purchase Rank 원재료',
            '선행연구_대응': 'Yang Liu(2025) monetary와 동일, '
                          '정박 효과 해석 추가'
        },
        'AOV': {
            '넛지_유형': '인지적 효율성(cognitive efficiency)',
            '이론적_근거': '한정적 합리성(bounded rationality) — '
                         'good enough 수준의 지출 기준점',
            '활용_목적': '평균 객단가 파악, 충동 구매 지수 라벨링 기준(1.5배)',
            '선행연구_대응': 'Yang Liu(2025) basket size 변수 확장, '
                          '인지적 구두쇠 해석 추가'
        },
        'Avg_Items_per_Order': {
            '넛지_유형': '유도성(affordance) / 인지적 효율성',
            '이론적_근거': '어포던스 — 추천 UI가 장바구니 확장 유도 vs '
                         '인지적 구두쇠의 단품 구매',
            '활용_목적': '탐색 vs 습관(목적) 구매 구분',
            '선행연구_대응': 'Yang Liu(2025) basket size와 유사, '
                          '이중 넛지 해석 추가'
        },
        'Active_Months': {
            '넛지_유형': '타성(inertia)',
            '이론적_근거': '디폴트 규칙(default rule) — '
                         '구매가 디폴트인 상태의 밀도(density)',
            '활용_목적': '구매 지속성 측정 (Purchase_Span과 차별: 밀도 vs 범위)',
            '선행연구_대응': 'Yang Liu(2025) tenure 변수와 유사, '
                          '타성 밀도 개념 추가'
        },
        'Purchase_Span': {
            '넛지_유형': '타성(inertia)',
            '이론적_근거': '현상유지 편향 — 구매 습관의 누적 범위(range)',
            '활용_목적': '고객 생애 길이 파악 (Active_Months와 차별: 범위 vs 밀도)',
            '선행연구_대응': 'Yang Liu(2025) tenure/cohort indicator와 유사, '
                          '타성 범위 개념 추가'
        }
    }

    # ── Step 7: 검증 출력 ────────────────────────────────────────
    if verbose:
        print(f"\n  생성된 고객 수: {len(rfm):,}명")
        print(f"  변수 목록: {list(rfm.columns[1:])}")
        print("-" * 65)
        print("  [넛지 유형 매핑 — 강준만(2016) 기반]")
        for var, info in nudge_mapping.items():
            print(f"    {var:.<25s} {info['넛지_유형']}")
        print("-" * 65)
        print("  [기술 통계]")
        print(rfm.describe().round(2).to_string())
        print("=" * 65)

    return rfm, nudge_mapping

In [8]:
rfm, nudge_map = build_category1_rfm(df_final)

카테고리 1: 기본 마케팅 변수 (RFM 계열) 생성 시작
  전체 데이터: 535,290행
  정상 주문만 필터링: 526,039행 (제외: 9,251행 — 반품·취소)

  생성된 고객 수: 6,516명
  변수 목록: ['Recency', 'Frequency', 'Monetary', 'AOV', 'Avg_Items_per_Order', 'Active_Months', 'Purchase_Span']
-----------------------------------------------------------------
  [넛지 유형 매핑 — 강준만(2016) 기반]
    Recency.................. 타성(inertia)
    Frequency................ 일관성(consistency)
    Monetary................. 비교성(comparability)
    AOV...................... 인지적 효율성(cognitive efficiency)
    Avg_Items_per_Order...... 유도성(affordance) / 인지적 효율성
    Active_Months............ 타성(inertia)
    Purchase_Span............ 타성(inertia)
-----------------------------------------------------------------
  [기술 통계]
       CustomerID  Recency  Frequency   Monetary       AOV  Avg_Items_per_Order  Active_Months  Purchase_Span
count     6516.00  6516.00    6516.00    6516.00   6516.00              6516.00        6516.00        6516.00
mean    496984.62   122.27       3.18    1575.

## 카테고리 2: 시간 패턴 변수 (타성 관련) 

In [9]:
from scipy.stats import entropy


def _normalized_entropy(series, n_categories):
    """
    정규화된 Shannon 엔트로피 (0~1 범위).
    - 0: 완전히 하나의 범주에 집중 (최대 타성)
    - 1: 모든 범주에 균등 분포 (타성 없음)

    Parameters
    ----------
    series : pd.Series
        범주형 시리즈 (예: 시간대, 요일, 월)
    n_categories : int
        가능한 범주 수 (시간=24, 요일=7, 월=12)
    """
    if len(series) <= 1:
        return 0.0
    prob = series.value_counts(normalize=True)
    raw = entropy(prob)
    max_ent = np.log(n_categories)
    return raw / max_ent if max_ent > 0 else 0.0


def build_category2_time_pattern(df_final, verbose=True):
    """
    df_final로부터 카테고리 2 시간 패턴 변수 9개를 생성한다.

    Parameters
    ----------
    df_final : pd.DataFrame
        전처리 완료 데이터 (반품 포함 상태).
        필수 컬럼: CustomerID, InvoiceNo, InvoiceDate
    verbose : bool
        True이면 생성 과정·기술통계·넛지 매핑 출력

    Returns
    -------
    time_features : pd.DataFrame
        고객별 9개 시간 패턴 변수 (CustomerID 포함)
    nudge_mapping : dict
        변수별 넛지 매핑 메타데이터
    """

    # ── Step 1: 정상 주문만 필터링 (카테고리 1과 동일 기준) ──────
    data = df_final.copy()
    normal = data[
        (~data['InvoiceNo'].astype(str).str.startswith('C')) &
        (data['Quantity'] > 0)
    ].copy()

    if verbose:
        print("=" * 65)
        print("카테고리 2: 시간 패턴 변수 (타성 관련) 생성 시작")
        print("=" * 65)
        print(f"  정상 주문 필터링: {len(normal):,}행")

    # ── Step 2: 시간 컴포넌트 추출 (원본 변경 없이 normal에만) ──
    normal['hour'] = normal['InvoiceDate'].dt.hour
    normal['dayofweek'] = normal['InvoiceDate'].dt.dayofweek
    normal['month'] = normal['InvoiceDate'].dt.month
    normal['is_weekend'] = normal['dayofweek'].isin([5, 6]).astype(int)

    # ── Step 3: 주문 단위 대표 시각 (IPI 계산용) ─────────────────
    #   같은 InvoiceNo의 여러 라인을 하나로 통합
    order_times = (
        normal
        .groupby(['CustomerID', 'InvoiceNo'])['InvoiceDate']
        .max()
        .reset_index()
        .sort_values(['CustomerID', 'InvoiceDate'])
    )

    # ── Step 4: 고객별 엔트로피 + Preferred Hour + Weekend Ratio ─
    def _agg_entropy(group):
        return pd.Series({
            'Hour_Entropy': _normalized_entropy(group['hour'], 24),
            'DayOfWeek_Entropy': _normalized_entropy(group['dayofweek'], 7),
            'Month_Entropy': _normalized_entropy(group['month'], 12),
            'Preferred_Hour': group['hour'].mode().iloc[0]
                              if not group['hour'].mode().empty else np.nan,
            'Weekend_Ratio': group['is_weekend'].mean(),
        })

    entropy_features = normal.groupby('CustomerID')[
        ['hour', 'dayofweek', 'month', 'is_weekend']
    ].apply(_agg_entropy)

    # ── Step 5: 구매 간격(IPI) 통계 ──────────────────────────────
    order_times['prev_date'] = order_times.groupby('CustomerID')['InvoiceDate'].shift(1)
    order_times['interval'] = (
        order_times['InvoiceDate'] - order_times['prev_date']
    ).dt.days

    ipi_stats = order_times.groupby('CustomerID')['interval'].agg(['mean', 'std'])
    ipi_stats.columns = ['IPI_Mean', 'IPI_Std']

    # CV: 표준편차 / 평균 (낮을수록 규칙적 = 타성 강함)
    # 1회 구매자 → IPI_Mean=NaN → CV=NaN (0으로 채우지 않음)
    ipi_stats['CV_Interval'] = ipi_stats['IPI_Std'] / ipi_stats['IPI_Mean']

    # ── Step 6: 결합 ─────────────────────────────────────────────
    time_features = entropy_features.join(ipi_stats, how='left')

    # 컬럼 순서 정리 (제안서 순서)
    time_features = time_features[[
        'Hour_Entropy', 'DayOfWeek_Entropy', 'Preferred_Hour',
        'Weekend_Ratio', 'IPI_Mean', 'IPI_Std', 'CV_Interval',
        'Month_Entropy'
    ]]
    time_features = time_features.reset_index()

    # ── Step 7: 결측 처리 전략 설명 ──────────────────────────────
    #   1회 구매자: IPI_Mean, IPI_Std, CV_Interval = NaN
    #   → fillna(0)하면 "완벽한 규칙성"으로 오해석되므로 NaN 유지
    #   → 이후 모델링 시 tree 기반 모델은 NaN 자체를 처리 가능
    #   → 또는 별도 플래그 변수(has_repeat_purchase)로 보완 가능
    n_nan_ipi = time_features['IPI_Mean'].isna().sum()

    if verbose:
        print(f"\n  생성된 고객 수: {len(time_features):,}명")
        print(f"  IPI 계산 불가 고객 (1회 구매): {n_nan_ipi:,}명 → NaN 유지")
        print(f"  변수 목록: {list(time_features.columns[1:])}")

    # ── Step 8: 넛지 유형 매핑 메타데이터 ────────────────────────
    nudge_mapping = {
        'Hour_Entropy': {
            '넛지_유형': '타성(inertia)',
            '이론적_근거': '현상유지 편향 — 낮을수록 특정 시간에 '
                         '고착된 디폴트 구매 패턴',
            '활용_목적': '구매 시간 앵커링 정도 측정, '
                       '종속변수 후보2(구매 맥락) 라벨링 기준',
            '선행연구_대응': 'Yang Liu(2025) temporal dummies를 '
                          '엔트로피로 연속 지표화, 타성 해석 추가',
            '정규화': '0~1 (24개 시간대 기준 정규화)'
        },
        'DayOfWeek_Entropy': {
            '넛지_유형': '타성(inertia)',
            '이론적_근거': '디폴트 규칙 — 특정 요일에 쇼핑하는 것이 '
                         '디폴트로 설정된 상태',
            '활용_목적': '요일 고착 정도 측정, '
                       '종속변수 후보2(구매 맥락) 라벨링 기준',
            '선행연구_대응': 'Yang Liu(2025) weekday dummy를 '
                          '엔트로피로 대체, 타성 밀도 해석 추가',
            '정규화': '0~1 (7개 요일 기준 정규화)'
        },
        'Preferred_Hour': {
            '넛지_유형': '타성(inertia)',
            '이론적_근거': '정박 효과(anchoring) — 가장 빈번한 구매 시간이 '
                         '이후 구매의 앵커로 작용',
            '활용_목적': '구매 앵커링 시점 파악 (범주형, 0~23)',
            '선행연구_대응': 'Yang Liu(2025)에 없는 변수, 본 연구 신규',
            '모델링_참고': '범주형 특성 → 순환 인코딩 또는 그룹화 고려'
        },
        'Weekend_Ratio': {
            '넛지_유형': '타성(inertia)',
            '이론적_근거': '현상유지 편향 — 주말/주중 쇼핑 습관이 '
                         '디폴트로 고착된 정도',
            '활용_목적': '라이프스타일 패턴 파악',
            '선행연구_대응': 'Yang Liu(2025) weekday/season dummy와 유사, '
                          '비율로 연속화하여 타성 해석 추가'
        },
        'IPI_Mean': {
            '넛지_유형': '타성(inertia)',
            '이론적_근거': '디폴트 규칙 — 평균 구매 주기 자체가 '
                         '고객의 디폴트 쇼핑 리듬',
            '활용_목적': '구매 주기 측정',
            '선행연구_대응': 'Yang Liu(2025) inter-purchase-time statistics와 동일, '
                          '디폴트 규칙 해석 추가',
            '결측_전략': '1회 구매자 → NaN 유지 (tree 모델 자체 처리)'
        },
        'IPI_Std': {
            '넛지_유형': '타성(inertia)',
            '이론적_근거': '현상유지 편향 — 낮을수록 일정한 주기 유지 = '
                         '타성이 강하게 작동',
            '활용_목적': '규칙성 vs 불규칙성 측정',
            '선행연구_대응': 'Yang Liu(2025) 동일 변수 존재, 타성 해석 추가',
            '결측_전략': '1회 구매자 → NaN 유지'
        },
        'CV_Interval': {
            '넛지_유형': '타성(inertia)',
            '이론적_근거': '현상유지 편향 — 스케일 독립적 리듬 안정성 지표, '
                         '낮을수록 타성 강함',
            '활용_목적': '구매 리듬 안정성 측정 (IPI_Mean 크기와 무관한 비교 가능)',
            '선행연구_대응': 'Yang Liu(2025)에 명시적 CV 없음, 본 연구 신규',
            '결측_전략': '1~2회 구매자 → NaN 유지'
        },
        'Month_Entropy': {
            '넛지_유형': '타성(inertia)',
            '이론적_근거': '현상유지 편향 — 낮을수록 특정 월(계절)에 '
                         '고착된 계절적 타성',
            '활용_목적': '계절성 패턴 파악',
            '선행연구_대응': 'Yang Liu(2025) season dummy를 '
                          '엔트로피로 대체, 계절적 타성 해석 추가',
            '정규화': '0~1 (12개월 기준 정규화)'
        }
    }

    # ── Step 9: 검증 출력 ────────────────────────────────────────
    if verbose:
        print("-" * 65)
        print("  [넛지 유형 매핑 — 강준만(2016) 타성(inertia)]")
        for var, info in nudge_mapping.items():
            label = info['넛지_유형']
            note = info.get('정규화', info.get('결측_전략', ''))
            suffix = f" | {note}" if note else ""
            print(f"    {var:.<25s} {label}{suffix}")
        print("-" * 65)
        print("  [기술 통계]")
        print(time_features.describe().round(4).to_string())
        print("=" * 65)

    return time_features, nudge_mapping

In [10]:
time_feat, nudge_map2 = build_category2_time_pattern(df_final)
features = pd.merge(rfm, time_feat, on='CustomerID', how='left')

카테고리 2: 시간 패턴 변수 (타성 관련) 생성 시작
  정상 주문 필터링: 526,039행

  생성된 고객 수: 6,516명
  IPI 계산 불가 고객 (1회 구매): 3,671명 → NaN 유지
  변수 목록: ['Hour_Entropy', 'DayOfWeek_Entropy', 'Preferred_Hour', 'Weekend_Ratio', 'IPI_Mean', 'IPI_Std', 'CV_Interval', 'Month_Entropy']
-----------------------------------------------------------------
  [넛지 유형 매핑 — 강준만(2016) 타성(inertia)]
    Hour_Entropy............. 타성(inertia) | 0~1 (24개 시간대 기준 정규화)
    DayOfWeek_Entropy........ 타성(inertia) | 0~1 (7개 요일 기준 정규화)
    Preferred_Hour........... 타성(inertia)
    Weekend_Ratio............ 타성(inertia)
    IPI_Mean................. 타성(inertia) | 1회 구매자 → NaN 유지 (tree 모델 자체 처리)
    IPI_Std.................. 타성(inertia) | 1회 구매자 → NaN 유지
    CV_Interval.............. 타성(inertia) | 1~2회 구매자 → NaN 유지
    Month_Entropy............ 타성(inertia) | 0~1 (12개월 기준 정규화)
-----------------------------------------------------------------
  [기술 통계]
         CustomerID  Hour_Entropy  DayOfWeek_Entropy  Preferred_Hour  Weekend_Ratio   IPI_Mean    I

## 카테고리 3: 장바구니 구성 변수 (인지적 효율성)

In [11]:
def build_category3_basket(df_final, verbose=True):
    """
    df_final로부터 카테고리 3 장바구니 구성 변수 6개를 생성한다.

    Parameters
    ----------
    df_final : pd.DataFrame
        전처리 완료 데이터 (반품 포함 상태).
        필수 컬럼: CustomerID, InvoiceNo, StockCode, Quantity, UnitPrice
    verbose : bool
        True이면 생성 과정·기술통계·넛지 매핑 출력

    Returns
    -------
    basket_features : pd.DataFrame
        고객별 6개 장바구니 변수 (CustomerID 포함)
    nudge_mapping : dict
        변수별 넛지 매핑 메타데이터
    """

    # ── Step 1: Revenue 생성 + 정상 주문 필터링 ──────────────────
    data = df_final.copy()
    data['Revenue'] = data['Quantity'] * data['UnitPrice']

    normal = data[
        (~data['InvoiceNo'].astype(str).str.startswith('C')) &
        (data['Quantity'] > 0) &
        (data['Revenue'] > 0)
    ].copy()

    if verbose:
        print("=" * 65)
        print("카테고리 3: 장바구니 구성 변수 (인지적 효율성) 생성 시작")
        print("=" * 65)
        print(f"  정상 주문 필터링: {len(normal):,}행")

    # ── Step 2: 주문(InvoiceNo) 단위 집계 ────────────────────────
    order_stats = normal.groupby(['CustomerID', 'InvoiceNo']).agg(
        unique_items=('StockCode', 'nunique'),
        total_qty=('Quantity', 'sum'),
        order_rev=('Revenue', 'sum')
    ).reset_index()

    # 주문 단위 파생 지표
    # Basket Diversity Index: 고유 품목 수 / 총 수량
    #   → 분모 0 방지 (정상 주문 필터 후에도 안전장치)
    order_stats['diversity_index'] = np.where(
        order_stats['total_qty'] > 0,
        order_stats['unique_items'] / order_stats['total_qty'],
        np.nan
    )

    # Large Basket 여부 (총 수량 > 10)
    order_stats['is_large'] = (order_stats['total_qty'] > 10).astype(int)

    # 단품 주문 여부 (고유 품목 수 == 1)
    order_stats['is_single'] = (order_stats['unique_items'] == 1).astype(int)

    # ── Step 3: 고객 단위 집계 → 6개 변수 ────────────────────────
    basket_features = order_stats.groupby('CustomerID').agg(
        Basket_Diversity_Index=('diversity_index', 'mean'),
        Avg_Basket_Size=('total_qty', 'mean'),
        Avg_Unique_Items_per_Basket=('unique_items', 'mean'),
        Large_Basket_Ratio=('is_large', 'mean'),
        Single_Item_Order_Ratio=('is_single', 'mean'),
        Max_Single_Order_Revenue=('order_rev', 'max')
    )

    # 컬럼 순서 정리 (제안서 순서)
    basket_features = basket_features[[
        'Basket_Diversity_Index', 'Avg_Basket_Size',
        'Avg_Unique_Items_per_Basket', 'Large_Basket_Ratio',
        'Single_Item_Order_Ratio', 'Max_Single_Order_Revenue'
    ]]
    basket_features = basket_features.reset_index()

    # ── Step 4: 넛지 유형 매핑 메타데이터 ────────────────────────
    nudge_mapping = {
        'Basket_Diversity_Index': {
            '넛지_유형': '인지적 효율성(cognitive efficiency)',
            '이론적_근거': '인지적 구두쇠 — 낮을수록 소수의 익숙한 제품에 '
                         '의존하여 인지 자원을 절약하는 상태',
            '활용_목적': '인지적 효율성 측정 (논문 제안서 핵심 변수)',
            '선행연구_대응': 'Yang Liu(2025) product-mix breadth와 유사, '
                          '인지적 구두쇠 해석 추가'
        },
        'Avg_Basket_Size': {
            '넛지_유형': '인지적 효율성(cognitive efficiency)',
            '이론적_근거': '한정적 합리성 — 대량 구매는 한 번의 의사결정으로 '
                         '인지 비용을 절약하는 satisficing 전략',
            '활용_목적': '구매 규모 파악',
            '선행연구_대응': 'Yang Liu(2025) basket size와 동일, '
                          'satisficing 해석 추가'
        },
        'Avg_Unique_Items_per_Basket': {
            '넛지_유형': '인지적 효율성(cognitive efficiency)',
            '이론적_근거': '가용성 편향 — 높을수록 다양한 선택지를 탐색하여 '
                         '인지 자원을 많이 투입하는 상태',
            '활용_목적': '탐색 vs 집중 구매 구분',
            '선행연구_대응': 'Yang Liu(2025)에 명시적 변수 없음, 본 연구 신규'
        },
        'Large_Basket_Ratio': {
            '넛지_유형': '인지적 효율성(cognitive efficiency)',
            '이론적_근거': '한정적 합리성 — 대량 구매 비율이 높을수록 '
                         '"한 번에 많이" 전략으로 의사결정 횟수 절약',
            '활용_목적': '도매·대량 구매 성향 파악',
            '선행연구_대응': 'Yang Liu(2025)에 없는 변수, 본 연구 신규'
        },
        'Single_Item_Order_Ratio': {
            '넛지_유형': '인지적 효율성(cognitive efficiency)',
            '이론적_근거': '인지적 구두쇠 — 단품 구매는 최소 인지 자원으로 '
                         '목적만 달성하는 극단적 효율 추구',
            '활용_목적': '목적 구매 성향 파악',
            '선행연구_대응': 'Yang Liu(2025)에 없는 변수, 본 연구 신규'
        },
        'Max_Single_Order_Revenue': {
            '넛지_유형': '인지적 효율성 / 흥미성(interest)',
            '이론적_근거': '한정적 합리성 이탈 — 평소 패턴 대비 비정상적 '
                         '대규모 구매는 인지적 효율성을 벗어난 충동·탐색 행동, '
                         '흥미성(재미 이론)과도 연결',
            '활용_목적': '충동·특별 구매 탐지, 종속변수 후보1(충동 구매 지수) 참고',
            '선행연구_대응': 'Yang Liu(2025) spending volatility 개념과 유사, '
                          '이중 넛지 해석 추가'
        }
    }

    # ── Step 5: 검증 출력 ────────────────────────────────────────
    if verbose:
        print(f"\n  생성된 고객 수: {len(basket_features):,}명")
        print(f"  변수 목록: {list(basket_features.columns[1:])}")
        print("-" * 65)
        print("  [넛지 유형 매핑 — 강준만(2016) 인지적 효율성]")
        for var, info in nudge_mapping.items():
            print(f"    {var:.<30s} {info['넛지_유형']}")
        print("-" * 65)
        print("  [기술 통계]")
        print(basket_features.describe().round(4).to_string())
        print("=" * 65)

    return basket_features, nudge_mapping

In [12]:
basket_feat, nudge_map3 = build_category3_basket(df_final)
features = features.merge(basket_feat, on='CustomerID', how='left')

카테고리 3: 장바구니 구성 변수 (인지적 효율성) 생성 시작
  정상 주문 필터링: 526,039행

  생성된 고객 수: 6,516명
  변수 목록: ['Basket_Diversity_Index', 'Avg_Basket_Size', 'Avg_Unique_Items_per_Basket', 'Large_Basket_Ratio', 'Single_Item_Order_Ratio', 'Max_Single_Order_Revenue']
-----------------------------------------------------------------
  [넛지 유형 매핑 — 강준만(2016) 인지적 효율성]
    Basket_Diversity_Index........ 인지적 효율성(cognitive efficiency)
    Avg_Basket_Size............... 인지적 효율성(cognitive efficiency)
    Avg_Unique_Items_per_Basket... 인지적 효율성(cognitive efficiency)
    Large_Basket_Ratio............ 인지적 효율성(cognitive efficiency)
    Single_Item_Order_Ratio....... 인지적 효율성(cognitive efficiency)
    Max_Single_Order_Revenue...... 인지적 효율성 / 흥미성(interest)
-----------------------------------------------------------------
  [기술 통계]
         CustomerID  Basket_Diversity_Index  Avg_Basket_Size  Avg_Unique_Items_per_Basket  Large_Basket_Ratio  Single_Item_Order_Ratio  Max_Single_Order_Revenue
count  6.516000e+03               6516.0

## 카테고리 4: 제품 충성도 변수

In [13]:
def build_category4_product_loyalty(df_final, verbose=True):
    """
    df_final로부터 카테고리 4 제품 충성도 변수 5개를 생성한다.

    Parameters
    ----------
    df_final : pd.DataFrame
        전처리 완료 데이터. 필수 컬럼:
        CustomerID, InvoiceNo, InvoiceDate, StockCode, Quantity, Description
    verbose : bool
        True이면 생성 과정·기술통계·넛지 매핑 출력

    Returns
    -------
    loyalty_features : pd.DataFrame
        고객별 5개 제품 충성도 변수 (CustomerID 포함)
    nudge_mapping : dict
        변수별 넛지 매핑 메타데이터
    """

    # ── Step 1: 정상 주문 필터링 ─────────────────────────────────
    data = df_final.copy()
    normal = data[
        (~data['InvoiceNo'].astype(str).str.startswith('C')) &
        (data['Quantity'] > 0)
    ].copy()

    if verbose:
        print("=" * 65)
        print("카테고리 4: 제품 충성도 변수 (유도성·타성) 생성 시작")
        print("=" * 65)
        print(f"  정상 주문 필터링: {len(normal):,}행")

    # ── Step 2: 카테고리 추출 ────────────────────────────────────
    #   Description 첫 단어는 "RED", "SET" 등 무의미할 수 있음
    #   → StockCode 앞 자리(숫자부)를 카테고리 프록시로 사용
    #   예: "85123A" → "851", "22423" → "224" (상위 3자리)
    normal['Category'] = (
        normal['StockCode']
        .astype(str)
        .str.extract(r'^(\d{1,3})', expand=False)
        .fillna('ETC')
    )

    if verbose:
        n_cats = normal['Category'].nunique()
        print(f"  카테고리 프록시 (StockCode 상위 3자리): {n_cats}개")

    # ── Step 3: 고객별 기초 통계 ─────────────────────────────────
    cust_stocks = normal.groupby('CustomerID')['StockCode']

    # 고유 품목 수
    unique_stocks = cust_stocks.nunique()
    # 총 구매 라인 수
    total_lines = cust_stocks.count()

    # ── Step 4: Repurchase Rate ──────────────────────────────────
    #   2회 이상 구매한 StockCode 수 / 전체 고유 품목 수
    def _repurchase_count(x):
        return (x.value_counts() >= 2).sum()

    repurch_count = cust_stocks.apply(_repurchase_count)
    repurchase_rate = repurch_count / unique_stocks
    repurchase_rate.name = 'Repurchase_Rate'

    # ── Step 5: Top Product Concentration ────────────────────────
    #   가장 많이 구매한 제품의 구매 비율 (라인 수 기준)
    def _top_concentration(x):
        return x.value_counts().iloc[0] / len(x) if len(x) > 0 else 0.0

    top_conc = cust_stocks.apply(_top_concentration)
    top_conc.name = 'Top_Product_Concentration'

    # ── Step 6: Unique Products Ratio ────────────────────────────
    #   고유 StockCode 수 / 총 구매 라인 수
    unique_ratio = unique_stocks / total_lines
    unique_ratio.name = 'Unique_Products_Ratio'

    # ── Step 7: Avg Repurchase Interval ──────────────────────────
    #   동일 제품(StockCode) 재구매 간격 평균 (일)
    sorted_data = normal.sort_values(['CustomerID', 'StockCode', 'InvoiceDate'])
    sorted_data['prev_date'] = (
        sorted_data
        .groupby(['CustomerID', 'StockCode'])['InvoiceDate']
        .shift(1)
    )
    sorted_data['repurch_interval'] = (
        sorted_data['InvoiceDate'] - sorted_data['prev_date']
    ).dt.days

    avg_repurch_interval = sorted_data.groupby('CustomerID')['repurch_interval'].mean()
    avg_repurch_interval.name = 'Avg_Repurchase_Interval'

    # ── Step 8: New Category Ratio ───────────────────────────────
    #   시계열상 신규 카테고리 첫 구매 비율
    #   (고객의 전체 구매 중 "이전에 사본 적 없는 카테고리" 비율)
    sorted_time = normal.sort_values(['CustomerID', 'InvoiceDate'])
    sorted_time['is_new_cat'] = ~sorted_time.duplicated(
        subset=['CustomerID', 'Category']
    )
    new_cat_ratio = sorted_time.groupby('CustomerID')['is_new_cat'].mean()
    new_cat_ratio.name = 'New_Category_Ratio'

    # ── Step 9: 결합 ─────────────────────────────────────────────
    loyalty_features = pd.concat(
        [repurchase_rate, top_conc, unique_ratio,
         avg_repurch_interval, new_cat_ratio],
        axis=1
    )

    # 컬럼 순서 정리 (제안서 순서)
    loyalty_features = loyalty_features[[
        'Repurchase_Rate', 'Top_Product_Concentration',
        'Unique_Products_Ratio', 'Avg_Repurchase_Interval',
        'New_Category_Ratio'
    ]]
    loyalty_features = loyalty_features.reset_index()

    # 결측 참고: Avg_Repurchase_Interval은 재구매 이력 없는 고객 → NaN
    n_nan = loyalty_features['Avg_Repurchase_Interval'].isna().sum()

    # ── Step 10: 넛지 유형 매핑 메타데이터 ───────────────────────
    nudge_mapping = {
        'Repurchase_Rate': {
            '넛지_유형': '유도성(affordance) / 타성(inertia)',
            '이론적_근거': '어포던스 — 추천 UI가 재구매를 유도하는 힘 + '
                         '현상유지 편향 — 익숙한 제품을 반복 선택하는 타성',
            '활용_목적': '유도성·타성 측정 (논문 제안서 핵심 변수)',
            '선행연구_대응': 'Yang Liu(2025)에 없는 변수, 본 연구 신규'
        },
        'Top_Product_Concentration': {
            '넛지_유형': '타성(inertia)',
            '이론적_근거': '현상유지 편향 — 높을수록 특정 제품에 고착, '
                         '소유 효과(endowment effect)로 익숙한 것에 집착',
            '활용_목적': '브랜드 고착 정도 파악',
            '선행연구_대응': 'Yang Liu(2025) product-mix breadth의 역지표, '
                          '소유 효과 해석 추가'
        },
        'Unique_Products_Ratio': {
            '넛지_유형': '유도성(affordance)',
            '이론적_근거': '어포던스 — 높을수록 다양한 제품을 탐색, '
                         '플랫폼 추천에 반응하는 유도성이 강한 상태',
            '활용_목적': '탐색 성향 측정',
            '선행연구_대응': 'Yang Liu(2025) product-mix breadth와 유사, '
                          '유도성 해석 추가'
        },
        'Avg_Repurchase_Interval': {
            '넛지_유형': '유도성(affordance)',
            '이론적_근거': '어포던스 반응 속도 — 짧을수록 유도에 빠르게 반응, '
                         '논문 제안서의 "거래 지름길 이용도" 개념',
            '활용_목적': '유도성 반응 속도 측정',
            '선행연구_대응': 'Yang Liu(2025)에 없는 변수, 본 연구 신규',
            '결측_전략': '재구매 이력 없는 고객 → NaN 유지'
        },
        'New_Category_Ratio': {
            '넛지_유형': '흥미성(interest)',
            '이론적_근거': '재미 이론(fun theory) — 새로운 카테고리 탐색은 '
                         '호기심·재미 충족 행동, 인정투쟁(새로운 자극 추구)',
            '활용_목적': '흥미성·탐색 성향 측정 (논문 제안서 핵심 변수)',
            '선행연구_대응': 'Yang Liu(2025)에 없는 변수, 본 연구 신규',
            '카테고리_정의': 'StockCode 상위 3자리를 카테고리 프록시로 사용'
        }
    }

    # ── Step 11: 검증 출력 ───────────────────────────────────────
    if verbose:
        print(f"\n  생성된 고객 수: {len(loyalty_features):,}명")
        print(f"  Avg_Repurchase_Interval NaN: {n_nan:,}명 → NaN 유지")
        print(f"  변수 목록: {list(loyalty_features.columns[1:])}")
        print("-" * 65)
        print("  [넛지 유형 매핑]")
        for var, info in nudge_mapping.items():
            print(f"    {var:.<30s} {info['넛지_유형']}")
        print("-" * 65)
        print("  [기술 통계]")
        print(loyalty_features.describe().round(4).to_string())
        print("=" * 65)

    return loyalty_features, nudge_mapping

In [14]:
loyalty_feat, nudge_map4 = build_category4_product_loyalty(df_final)
features = features.merge(loyalty_feat, on='CustomerID', how='left')

카테고리 4: 제품 충성도 변수 (유도성·타성) 생성 시작
  정상 주문 필터링: 526,039행
  카테고리 프록시 (StockCode 상위 3자리): 112개

  생성된 고객 수: 6,516명
  Avg_Repurchase_Interval NaN: 3,562명 → NaN 유지
  변수 목록: ['Repurchase_Rate', 'Top_Product_Concentration', 'Unique_Products_Ratio', 'Avg_Repurchase_Interval', 'New_Category_Ratio']
-----------------------------------------------------------------
  [넛지 유형 매핑]
    Repurchase_Rate............... 유도성(affordance) / 타성(inertia)
    Top_Product_Concentration..... 타성(inertia)
    Unique_Products_Ratio......... 유도성(affordance)
    Avg_Repurchase_Interval....... 유도성(affordance)
    New_Category_Ratio............ 흥미성(interest)
-----------------------------------------------------------------
  [기술 통계]
         CustomerID  Repurchase_Rate  Top_Product_Concentration  Unique_Products_Ratio  Avg_Repurchase_Interval  New_Category_Ratio
count  6.516000e+03        6516.0000                  6516.0000              6516.0000                2954.0000           6516.0000
mean   4.969846e+05         

## 카테고리 5: 가격 민감도 변수 (긍정성 관련)

In [15]:
def build_category5_price(df_final, split_date='2011-06-10', verbose=True):
    """
    df_final로부터 카테고리 5 가격 민감도 변수 6개를 생성한다.

    Parameters
    ----------
    df_final : pd.DataFrame
        전처리 완료 데이터.
        필수 컬럼: CustomerID, InvoiceNo, InvoiceDate, Quantity, UnitPrice
    split_date : str
        Revenue Trend 계산용 전반/후반 분기점 (기본값: '2011-06-10')
    verbose : bool
        True이면 생성 과정·기술통계·넛지 매핑 출력

    Returns
    -------
    price_features : pd.DataFrame
        고객별 6개 가격 민감도 변수 (CustomerID 포함)
    nudge_mapping : dict
        변수별 넛지 매핑 메타데이터
    """

    split_dt = pd.to_datetime(split_date)

    # ── Step 1: Revenue 생성 + 정상 주문 필터링 ──────────────────
    data = df_final.copy()
    data['Revenue'] = data['Quantity'] * data['UnitPrice']

    normal = data[
        (~data['InvoiceNo'].astype(str).str.startswith('C')) &
        (data['Quantity'] > 0) &
        (data['Revenue'] > 0)
    ].copy()

    if verbose:
        print("=" * 65)
        print("카테고리 5: 가격 민감도 변수 (긍정성) 생성 시작")
        print("=" * 65)
        print(f"  정상 주문 필터링: {len(normal):,}행")

    # ── Step 2: 단가 기준치 (정상 주문 기준) ─────────────────────
    q1_price = normal['UnitPrice'].quantile(0.25)
    q3_price = normal['UnitPrice'].quantile(0.75)

    if verbose:
        print(f"  단가 기준: 하위 25% ≤ £{q1_price:.2f} / "
              f"상위 25% ≥ £{q3_price:.2f}")

    # ── Step 3: 고객별 기본 가격 지표 ────────────────────────────
    avg_price = normal.groupby('CustomerID')['UnitPrice'].mean()
    avg_price.name = 'Avg_Unit_Price'

    price_var = normal.groupby('CustomerID')['UnitPrice'].std()
    price_var.name = 'Price_Variance'

    # ── Step 4: 고단가/저단가 비율 (원본 변경 없이) ───────────────
    normal['is_high'] = (normal['UnitPrice'] >= q3_price).astype(int)
    normal['is_low'] = (normal['UnitPrice'] <= q1_price).astype(int)

    high_ratio = normal.groupby('CustomerID')['is_high'].mean()
    high_ratio.name = 'High_Price_Item_Ratio'

    low_ratio = normal.groupby('CustomerID')['is_low'].mean()
    low_ratio.name = 'Low_Price_Item_Ratio'

    # ── Step 5: Revenue per Visit (함수 내 자체 계산) ─────────────
    order_rev = normal.groupby(['CustomerID', 'InvoiceNo'])['Revenue'].sum()
    rev_per_visit = order_rev.groupby('CustomerID').mean()
    rev_per_visit.name = 'Revenue_per_Visit'

    # ── Step 6: Revenue Trend (후반부 / 전반부) ───────────────────
    first_half = (
        normal[normal['InvoiceDate'] < split_dt]
        .groupby('CustomerID')['Revenue'].sum()
    )
    second_half = (
        normal[normal['InvoiceDate'] >= split_dt]
        .groupby('CustomerID')['Revenue'].sum()
    )

    # 전반부만 있는 고객: 후반부 매출 0 → trend = 0 (이탈 방향)
    # 후반부만 있는 고객: 전반부 매출 0 → trend = inf → NaN 처리
    # 양쪽 다 있는 고객: 정상 비율 계산
    revenue_trend = (second_half / first_half).replace(
        [np.inf, -np.inf], np.nan
    )
    revenue_trend.name = 'Revenue_Trend'

    # ── Step 7: 결합 ─────────────────────────────────────────────
    price_features = pd.concat(
        [avg_price, price_var, high_ratio, low_ratio,
         rev_per_visit, revenue_trend],
        axis=1
    )

    # 컬럼 순서 정리 (제안서 순서)
    price_features = price_features[[
        'Avg_Unit_Price', 'Price_Variance',
        'High_Price_Item_Ratio', 'Low_Price_Item_Ratio',
        'Revenue_per_Visit', 'Revenue_Trend'
    ]]
    price_features = price_features.reset_index()

    # 결측 참고
    n_nan_var = price_features['Price_Variance'].isna().sum()
    n_nan_trend = price_features['Revenue_Trend'].isna().sum()

    # ── Step 8: 넛지 유형 매핑 메타데이터 ────────────────────────
    nudge_mapping = {
        'Avg_Unit_Price': {
            '넛지_유형': '긍정성(positivity)',
            '이론적_근거': '프레임 이론 — 높은 가격대 선호는 '
                         '"좋은 것을 선택하는 나"라는 긍정적 자기 프레이밍',
            '활용_목적': '가격대 선호 파악',
            '선행연구_대응': 'Yang Liu(2025) price dispersion 관련, '
                          '긍정성 프레임 해석 추가'
        },
        'Price_Variance': {
            '넛지_유형': '긍정성(positivity) / 인지적 효율성',
            '이론적_근거': '프레임 이론 — 낮을수록 일정한 가격대에 '
                         '머무르는 안정적 자기상, 높을수록 탐색적 소비',
            '활용_목적': '가격 일관성 측정',
            '선행연구_대응': 'Yang Liu(2025) spending volatility와 유사, '
                          '긍정성 안정성 해석 추가',
            '결측_전략': '1회 구매자 → NaN 유지'
        },
        'High_Price_Item_Ratio': {
            '넛지_유형': '긍정성(positivity)',
            '이론적_근거': '자기이행적 예언 — 고단가 구매 비율이 높을수록 '
                         '"나는 좋은 것을 살 자격이 있다"는 긍정적 프레임 강화',
            '활용_목적': '탐색적 vs 습관적 구매 구분',
            '선행연구_대응': 'Yang Liu(2025)에 없는 변수, 본 연구 신규'
        },
        'Low_Price_Item_Ratio': {
            '넛지_유형': '긍정성(positivity) / 타성(inertia)',
            '이론적_근거': '학습된 무력감 — 저단가 제품만 반복 구매하는 패턴은 '
                         '"어차피 비싼 건 안 사" 라는 학습된 무력감의 행동 지표, '
                         '동시에 저관여 습관 구매로서의 타성',
            '활용_목적': '저관여 습관 구매 측정',
            '선행연구_대응': 'Yang Liu(2025)에 없는 변수, 본 연구 신규'
        },
        'Revenue_per_Visit': {
            '넛지_유형': '긍정성(positivity)',
            '이론적_근거': '프레임 이론 — 방문당 지출 강도는 '
                         '구매 경험의 긍정성 수준을 반영',
            '활용_목적': '지출 강도 측정',
            '선행연구_대응': 'Yang Liu(2025) monetary/frequency 비율과 유사, '
                          '긍정성 강도 해석 추가'
        },
        'Revenue_Trend': {
            '넛지_유형': '긍정성(positivity)',
            '이론적_근거': '자기이행적 예언 — 상승 추세는 긍정적 소비 경험이 '
                         '다음 구매를 강화하는 선순환, '
                         '하락 추세는 부정적 경험의 악순환(학습된 무력감)',
            '활용_목적': '구매 성장·이탈 방향 탐지',
            '선행연구_대응': 'Yang Liu(2025)에 없는 변수, 본 연구 신규',
            '결측_전략': '후반부만 구매(신규 고객) → NaN 유지'
        }
    }

    # ── Step 9: 검증 출력 ────────────────────────────────────────
    if verbose:
        print(f"\n  생성된 고객 수: {len(price_features):,}명")
        print(f"  Price_Variance NaN (1회 구매): {n_nan_var:,}명")
        print(f"  Revenue_Trend NaN (후반부만 구매): {n_nan_trend:,}명")
        print(f"  변수 목록: {list(price_features.columns[1:])}")
        print("-" * 65)
        print("  [넛지 유형 매핑 — 강준만(2016) 긍정성(positivity)]")
        for var, info in nudge_mapping.items():
            print(f"    {var:.<25s} {info['넛지_유형']}")
        print("-" * 65)
        print("  [기술 통계]")
        print(price_features.describe().round(4).to_string())
        print("=" * 65)

    return price_features, nudge_mapping

In [16]:
price_feat, nudge_map5 = build_category5_price(df_final)
features = features.merge(price_feat, on='CustomerID', how='left')

카테고리 5: 가격 민감도 변수 (긍정성) 생성 시작
  정상 주문 필터링: 526,039행
  단가 기준: 하위 25% ≤ £1.25 / 상위 25% ≥ £4.13

  생성된 고객 수: 6,516명
  Price_Variance NaN (1회 구매): 1,040명
  Revenue_Trend NaN (후반부만 구매): 4,568명
  변수 목록: ['Avg_Unit_Price', 'Price_Variance', 'High_Price_Item_Ratio', 'Low_Price_Item_Ratio', 'Revenue_per_Visit', 'Revenue_Trend']
-----------------------------------------------------------------
  [넛지 유형 매핑 — 강준만(2016) 긍정성(positivity)]
    Avg_Unit_Price........... 긍정성(positivity)
    Price_Variance........... 긍정성(positivity) / 인지적 효율성
    High_Price_Item_Ratio.... 긍정성(positivity)
    Low_Price_Item_Ratio..... 긍정성(positivity) / 타성(inertia)
    Revenue_per_Visit........ 긍정성(positivity)
    Revenue_Trend............ 긍정성(positivity)
-----------------------------------------------------------------
  [기술 통계]
         CustomerID  Avg_Unit_Price  Price_Variance  High_Price_Item_Ratio  Low_Price_Item_Ratio  Revenue_per_Visit  Revenue_Trend
count  6.516000e+03       6516.0000       5476.0000              

## 카테고리 6: 반품·취소 행동 변수 (긍정성)

In [17]:
def build_category6_return(df_final, verbose=True):
    """
    df_final로부터 카테고리 6 반품·취소 행동 변수 5개를 생성한다.

    Parameters
    ----------
    df_final : pd.DataFrame
        전처리 완료 데이터 (반품 포함 상태).
        필수 컬럼: CustomerID, InvoiceNo, InvoiceDate, Quantity
    verbose : bool
        True이면 생성 과정·기술통계·넛지 매핑 출력

    Returns
    -------
    return_features : pd.DataFrame
        고객별 5개 반품·취소 변수 (CustomerID 포함)
    nudge_mapping : dict
        변수별 넛지 매핑 메타데이터
    """

    data = df_final.copy()

    if verbose:
        print("=" * 65)
        print("카테고리 6: 반품·취소 행동 변수 (긍정성) 생성 시작")
        print("=" * 65)

    # ── Step 1: 주문 유형 분류 ───────────────────────────────────
    data['is_cancel'] = data['InvoiceNo'].astype(str).str.startswith('C')
    data['is_return'] = (data['Quantity'] < 0)

    # 정상 주문만
    normal = data[~data['is_cancel'] & (data['Quantity'] > 0)].copy()
    # 반품 건 (Quantity < 0, C 접두사 포함)
    returns = data[data['is_return'] | data['is_cancel']].copy()

    if verbose:
        print(f"  정상 주문: {len(normal):,}행")
        print(f"  반품·취소: {len(returns):,}행")

    # ── Step 2: 고객별 주문 수 (정상 기준 분모) ──────────────────
    normal_orders = normal.groupby('CustomerID')['InvoiceNo'].nunique()
    normal_orders.name = 'normal_order_count'

    # 전체 고객 목록 확보
    all_customers = data['CustomerID'].unique()

    # ── Step 3: Return Rate ──────────────────────────────────────
    #   반품 건수(Quantity < 0인 라인 수) / 전체 정상 주문 수
    return_line_count = returns.groupby('CustomerID').size()
    return_rate = (return_line_count / normal_orders).reindex(all_customers).fillna(0)
    return_rate.name = 'Return_Rate'

    # ── Step 4: Has Return ───────────────────────────────────────
    has_return = (return_rate > 0).astype(int)
    has_return.name = 'Has_Return'

    # ── Step 5: Cancellation Rate ────────────────────────────────
    #   C로 시작하는 InvoiceNo 수 / 전체 고유 InvoiceNo 수
    all_invoices = data.groupby('CustomerID')['InvoiceNo'].nunique()
    cancel_invoices = (
        data[data['is_cancel']]
        .groupby('CustomerID')['InvoiceNo'].nunique()
    )
    cancel_rate = (cancel_invoices / all_invoices).reindex(all_customers).fillna(0)
    cancel_rate.name = 'Cancellation_Rate'

    # ── Step 6: Days to Return Purchase & Recovery Ratio ─────────
    #   벡터화 방식으로 최적화 (고객별 loop 제거)

    # 6-1. 반품 고객의 반품 시점 (고객별 가장 마지막 반품일)
    return_dates = (
        returns
        .groupby('CustomerID')['InvoiceDate']
        .max()
        .rename('last_return_date')
    )

    # 6-2. 정상 주문 시점 테이블
    normal_dates = normal[['CustomerID', 'InvoiceDate']].drop_duplicates()

    # 6-3. 반품 고객과 정상 주문 조인
    recovery_df = return_dates.reset_index().merge(
        normal_dates, on='CustomerID', how='inner'
    )

    # 6-4. 반품일 이후의 정상 구매만 필터
    recovery_df = recovery_df[
        recovery_df['InvoiceDate'] > recovery_df['last_return_date']
    ].copy()

    # 6-5. 반품 후 가장 빠른 정상 구매까지의 일수
    recovery_df['days_gap'] = (
        recovery_df['InvoiceDate'] - recovery_df['last_return_date']
    ).dt.days

    days_to_return = recovery_df.groupby('CustomerID')['days_gap'].min()
    days_to_return.name = 'Days_to_Return_Purchase'

    # 6-6. Return Recovery Ratio: 반품 후 재진입 여부 (0/1)
    return_customers = set(return_dates.index)
    recovered_customers = set(days_to_return.index)

    recovery_ratio = pd.Series(0, index=all_customers, name='Return_Recovery_Ratio')
    for cid in recovered_customers:
        recovery_ratio[cid] = 1

    # 반품 경험이 없는 고객은 NaN이 아닌 0 (반품 자체가 없으므로)
    # 반품 경험이 있지만 재진입하지 않은 고객도 0

    # ── Step 7: 결합 ─────────────────────────────────────────────
    return_features = pd.DataFrame({
        'Return_Rate': return_rate,
        'Has_Return': has_return,
        'Days_to_Return_Purchase': days_to_return,
        'Cancellation_Rate': cancel_rate,
        'Return_Recovery_Ratio': recovery_ratio
    }, index=all_customers)

    # 컬럼 순서 정리 (제안서 순서)
    return_features = return_features[[
        'Return_Rate', 'Has_Return', 'Days_to_Return_Purchase',
        'Cancellation_Rate', 'Return_Recovery_Ratio'
    ]]
    return_features.index.name = 'CustomerID'
    return_features = return_features.reset_index()

    # 결측 참고
    n_has_return = (return_features['Has_Return'] == 1).sum()
    n_recovered = (return_features['Return_Recovery_Ratio'] == 1).sum()
    n_nan_days = return_features['Days_to_Return_Purchase'].isna().sum()

    # ── Step 8: 넛지 유형 매핑 메타데이터 ────────────────────────
    nudge_mapping = {
        'Return_Rate': {
            '넛지_유형': '긍정성(positivity)',
            '이론적_근거': '프레임 이론 — 반품률은 부정적 소비 경험의 '
                         '빈도 지표, 높을수록 긍정성 프레임 약화',
            '활용_목적': '긍정성·신뢰도 측정',
            '선행연구_대응': 'Yang Liu(2025) return ratio와 동일, '
                          '긍정성 프레임 해석 추가'
        },
        'Has_Return': {
            '넛지_유형': '긍정성(positivity)',
            '이론적_근거': '프레임 이론 — 반품 경험 유무 자체가 '
                         '소비 프레임의 긍정/부정 분기점',
            '활용_목적': '이진 지표 (반품 경험 여부)',
            '선행연구_대응': 'Yang Liu(2025)에 명시적 변수 없음, 본 연구 신규'
        },
        'Days_to_Return_Purchase': {
            '넛지_유형': '긍정성(positivity)',
            '이론적_근거': '자기이행적 예언 — 반품 후 빠른 재진입은 '
                         '"이 가게는 괜찮다"는 긍정적 프레임 회복, '
                         '느린 재진입은 학습된 무력감으로 이탈 방향 '
                         '(논문 제안서 핵심 변수)',
            '활용_목적': '긍정성 회복 속도 측정',
            '선행연구_대응': 'Yang Liu(2025)에 없는 변수, 본 연구 신규',
            '결측_전략': '반품 미경험 또는 미회복 고객 → NaN 유지'
        },
        'Cancellation_Rate': {
            '넛지_유형': '긍정성(positivity)',
            '이론적_근거': '학습된 무력감 — 주문 취소 빈도가 높을수록 '
                         '구매 결정에 대한 자신감(긍정성) 부족',
            '활용_목적': '구매 결정 불안정성 측정',
            '선행연구_대응': 'Yang Liu(2025) "remove administrative '
                          'cancellations"으로 제거한 데이터를 본 연구는 '
                          '피처로 활용 — 핵심 차별점'
        },
        'Return_Recovery_Ratio': {
            '넛지_유형': '긍정성(positivity)',
            '이론적_근거': '자기이행적 예언 — 반품 후 재진입 여부는 '
                         '긍정적 소비 경험 회복의 최종 결과 지표',
            '활용_목적': '긍정성 넛지 효과 측정',
            '선행연구_대응': 'Yang Liu(2025)에 없는 변수, 본 연구 신규'
        }
    }

    # ── Step 9: 검증 출력 ────────────────────────────────────────
    if verbose:
        print(f"\n  생성된 고객 수: {len(return_features):,}명")
        print(f"  반품 경험 고객: {n_has_return:,}명")
        print(f"  반품 후 재진입 고객: {n_recovered:,}명")
        print(f"  Days_to_Return_Purchase NaN: {n_nan_days:,}명")
        print(f"  변수 목록: {list(return_features.columns[1:])}")
        print("-" * 65)
        print("  [넛지 유형 매핑 — 강준만(2016) 긍정성(positivity)]")
        for var, info in nudge_mapping.items():
            print(f"    {var:.<28s} {info['넛지_유형']}")
        print("-" * 65)
        print("  [기술 통계]")
        print(return_features.describe().round(4).to_string())
        print("=" * 65)

    return return_features, nudge_mapping

In [18]:
return_feat, nudge_map6 = build_category6_return(df_final)
features = features.merge(return_feat, on='CustomerID', how='left')

카테고리 6: 반품·취소 행동 변수 (긍정성) 생성 시작
  정상 주문: 526,039행
  반품·취소: 9,251행

  생성된 고객 수: 6,731명
  반품 경험 고객: 1,556명
  반품 후 재진입 고객: 1,046명
  Days_to_Return_Purchase NaN: 5,685명
  변수 목록: ['Return_Rate', 'Has_Return', 'Days_to_Return_Purchase', 'Cancellation_Rate', 'Return_Recovery_Ratio']
-----------------------------------------------------------------
  [넛지 유형 매핑 — 강준만(2016) 긍정성(positivity)]
    Return_Rate................. 긍정성(positivity)
    Has_Return.................. 긍정성(positivity)
    Days_to_Return_Purchase..... 긍정성(positivity)
    Cancellation_Rate........... 긍정성(positivity)
    Return_Recovery_Ratio....... 긍정성(positivity)
-----------------------------------------------------------------
  [기술 통계]
         CustomerID  Return_Rate  Has_Return  Days_to_Return_Purchase  Cancellation_Rate  Return_Recovery_Ratio
count  6.731000e+03    6731.0000   6731.0000                1046.0000          6731.0000              6731.0000
mean   5.205769e+05       0.2759      0.2312                  44.9493  

## 카테고리 7: 사회적 비교 변수 (비교성)

In [19]:
def build_category7_social(df_final, top_pct=0.10, verbose=True):
    """
    df_final로부터 카테고리 7 사회적 비교 변수 3개를 생성한다.

    Parameters
    ----------
    df_final : pd.DataFrame
        전처리 완료 데이터.
        필수 컬럼: CustomerID, InvoiceNo, InvoiceDate, StockCode,
                  Quantity, UnitPrice, Country
    top_pct : float
        국가별 인기 품목 기준 (기본값: 상위 10%)
    verbose : bool
        True이면 생성 과정·기술통계·넛지 매핑 출력

    Returns
    -------
    social_features : pd.DataFrame
        고객별 3개 사회적 비교 변수 (CustomerID 포함)
    nudge_mapping : dict
        변수별 넛지 매핑 메타데이터
    """

    # ── Step 1: Revenue 생성 + 정상 주문 필터링 ──────────────────
    data = df_final.copy()
    data['Revenue'] = data['Quantity'] * data['UnitPrice']

    normal = data[
        (~data['InvoiceNo'].astype(str).str.startswith('C')) &
        (data['Quantity'] > 0) &
        (data['Revenue'] > 0)
    ].copy()

    if verbose:
        print("=" * 65)
        print("카테고리 7: 사회적 비교 변수 (비교성) 생성 시작")
        print("=" * 65)
        print(f"  정상 주문 필터링: {len(normal):,}행")

    # ── Step 2: 고객별 국가 매핑 + UK Flag ───────────────────────
    cust_country = normal.groupby('CustomerID')['Country'].first()
    cust_country.name = 'Country'

    uk_flag = (cust_country == 'United Kingdom').astype(int)
    uk_flag.name = 'UK_Flag'

    if verbose:
        n_countries = cust_country.nunique()
        n_uk = (uk_flag == 1).sum()
        print(f"  국가 수: {n_countries}개 (UK: {n_uk:,}명)")

    # ── Step 3: Country Trend Alignment (벡터화) ─────────────────
    #   국가별 인기 품목(상위 10%) 집합 구축
    country_item_counts = (
        normal.groupby(['Country', 'StockCode'])
        .size()
        .reset_index(name='count')
    )

    def _get_top_items(group):
        n = max(1, int(len(group) * top_pct))
        return set(group.nlargest(n, 'count')['StockCode'])

    country_trends = (
        country_item_counts
        .groupby('Country')[['StockCode', 'count']]
        .apply(_get_top_items)
        .to_dict()
    )

    #   고객별 구매 품목 집합 구축
    cust_items = normal.groupby('CustomerID')['StockCode'].apply(set)

    #   일치율 계산 (벡터화)
    def _calc_alignment(cid):
        country = cust_country.get(cid)
        items = cust_items.get(cid, set())
        trends = country_trends.get(country, set())
        if not items or not trends:
            return 0.0
        return len(items & trends) / len(items)

    alignment = pd.Series(
        {cid: _calc_alignment(cid) for cid in cust_country.index},
        name='Country_Trend_Alignment'
    )

    # ── Step 4: Country Purchase Rank (국가 내 매출 백분위) ───────
    #   함수 내 자체 계산 (외부 rfm_features 의존 제거)
    cust_monetary = normal.groupby('CustomerID')['Revenue'].sum()
    cust_monetary.name = 'Monetary'

    rank_df = pd.DataFrame({
        'Country': cust_country,
        'Monetary': cust_monetary
    })

    rank_df['Country_Purchase_Rank'] = (
        rank_df.groupby('Country')['Monetary']
        .rank(pct=True)
    )

    country_rank = rank_df['Country_Purchase_Rank']
    country_rank.name = 'Country_Purchase_Rank'

    # ── Step 5: 결합 ─────────────────────────────────────────────
    social_features = pd.concat(
        [alignment, country_rank, uk_flag],
        axis=1
    )

    social_features = social_features[[
        'Country_Trend_Alignment', 'Country_Purchase_Rank', 'UK_Flag'
    ]]
    social_features.index.name = 'CustomerID'
    social_features = social_features.reset_index()

    # ── Step 6: 넛지 유형 매핑 메타데이터 ────────────────────────
    nudge_mapping = {
        'Country_Trend_Alignment': {
            '넛지_유형': '비교성(comparability)',
            '이론적_근거': '사회적 증거(social proof) — 거주 국가의 인기 품목과 '
                         '개인 장바구니 일치율이 높을수록 '
                         '"다른 사람들이 사는 것을 따라 사는" 사회적 증거 행동 '
                         '(논문 제안서 핵심 변수)',
            '활용_목적': '비교성 측정',
            '선행연구_대응': 'Yang Liu(2025)에 없는 변수, 본 연구 신규'
        },
        'Country_Purchase_Rank': {
            '넛지_유형': '비교성(comparability)',
            '이론적_근거': '사회비교 이론 — 동일 국가 내 지출 백분위는 '
                         '"나는 다른 사람들보다 얼마나 소비하는가"의 '
                         '사회적 위치 지표, 정박 효과의 기준점',
            '활용_목적': '사회적 위치 파악',
            '선행연구_대응': 'Yang Liu(2025)에 없는 변수, 본 연구 신규'
        },
        'UK_Flag': {
            '넛지_유형': '비교성(comparability)',
            '이론적_근거': '사회비교 이론 — UK(본사 소재국) vs 해외 고객의 '
                         '구매 행동 차이는 사회적 맥락(proximity)에 의한 것, '
                         '시장 세분화 기준 변수',
            '활용_목적': '시장 세분화 기준 변수',
            '선행연구_대응': 'Yang Liu(2025)에 없는 변수, 본 연구 신규'
        }
    }

    # ── Step 7: 검증 출력 ────────────────────────────────────────
    if verbose:
        print(f"\n  생성된 고객 수: {len(social_features):,}명")
        print(f"  변수 목록: {list(social_features.columns[1:])}")
        print("-" * 65)
        print("  [넛지 유형 매핑 — 강준만(2016) 비교성(comparability)]")
        for var, info in nudge_mapping.items():
            print(f"    {var:.<28s} {info['넛지_유형']}")
        print("-" * 65)
        print("  [기술 통계]")
        print(social_features.describe().round(4).to_string())
        print("=" * 65)

    return social_features, nudge_mapping


# ── 사용 예시 ────────────────────────────────────────────────────
# social_feat, nudge_map7 = build_category7_social(df_final)
# features = features.merge(social_feat, on='CustomerID', how='left')

In [20]:
social_feat, nudge_map7 = build_category7_social(df_final)
features = features.merge(social_feat, on='CustomerID', how='left')

카테고리 7: 사회적 비교 변수 (비교성) 생성 시작
  정상 주문 필터링: 526,039행
  국가 수: 38개 (UK: 6,043명)

  생성된 고객 수: 6,516명
  변수 목록: ['Country_Trend_Alignment', 'Country_Purchase_Rank', 'UK_Flag']
-----------------------------------------------------------------
  [넛지 유형 매핑 — 강준만(2016) 비교성(comparability)]
    Country_Trend_Alignment..... 비교성(comparability)
    Country_Purchase_Rank....... 비교성(comparability)
    UK_Flag..................... 비교성(comparability)
-----------------------------------------------------------------
  [기술 통계]
         CustomerID  Country_Trend_Alignment  Country_Purchase_Rank    UK_Flag
count  6.516000e+03                6516.0000              6516.0000  6516.0000
mean   4.969846e+05                   0.3929                 0.5029     0.9274
std    6.801235e+05                   0.2544                 0.2892     0.2595
min    1.234600e+04                   0.0000                 0.0106     0.0000
25%    1.455475e+04                   0.2500                 0.2517     1.0000
50%    1.67830

## 카테고리 8: 일관성 변수 (일관성 넛지)

In [21]:
def build_category8_consistency(df_final, verbose=True):
    """
    df_final로부터 카테고리 8 일관성 변수 3개를 생성한다.

    Parameters
    ----------
    df_final : pd.DataFrame
        전처리 완료 데이터.
        필수 컬럼: CustomerID, InvoiceNo, InvoiceDate, StockCode,
                  Quantity, UnitPrice
    verbose : bool
        True이면 생성 과정·기술통계·넛지 매핑 출력

    Returns
    -------
    consistency_features : pd.DataFrame
        고객별 3개 일관성 변수 (CustomerID 포함)
    nudge_mapping : dict
        변수별 넛지 매핑 메타데이터
    """

    # ── Step 1: Revenue 생성 + 정상 주문 필터링 ──────────────────
    data = df_final.copy()
    data['Revenue'] = data['Quantity'] * data['UnitPrice']

    normal = data[
        (~data['InvoiceNo'].astype(str).str.startswith('C')) &
        (data['Quantity'] > 0) &
        (data['Revenue'] > 0)
    ].copy()

    if verbose:
        print("=" * 65)
        print("카테고리 8: 일관성 변수 (일관성 넛지) 생성 시작")
        print("=" * 65)
        print(f"  정상 주문 필터링: {len(normal):,}행")

    # ── Step 2: 카테고리 프록시 (카테고리 4와 동일 방식) ──────────
    normal['Category'] = (
        normal['StockCode']
        .astype(str)
        .str.extract(r'^(\d{1,3})', expand=False)
        .fillna('ETC')
    )

    # ── Step 3: Category Concentration Std ───────────────────────
    #   월별 카테고리 점유율의 표준편차 → 낮을수록 일관된 취향
    normal['month_year'] = normal['InvoiceDate'].dt.to_period('M')

    cat_monthly = (
        normal
        .groupby(['CustomerID', 'month_year', 'Category'])
        .size()
        .unstack(fill_value=0)
    )

    # 월별 점유율 계산
    cat_share = cat_monthly.div(cat_monthly.sum(axis=1), axis=0)

    # 고객별: 각 카테고리 점유율의 시계열 표준편차 → 평균
    # 2개월 이상 데이터가 있는 고객만 의미 있음
    cat_std = cat_share.groupby(level=0).std().mean(axis=1)
    cat_std.name = 'Category_Concentration_Std'

    # ── Step 4: Revenue Consistency (주문별 Revenue의 CV) ────────
    #   CV = Std / Mean → 낮을수록 지출 일관성 높음
    order_rev = (
        normal
        .groupby(['CustomerID', 'InvoiceNo'])['Revenue']
        .sum()
    )

    rev_stats = order_rev.groupby('CustomerID').agg(['mean', 'std'])
    rev_cv = rev_stats['std'] / rev_stats['mean']
    rev_cv.name = 'Revenue_Consistency'

    # ── Step 5: Purchase Rhythm Score (구매 간격 CV의 역수) ───────
    #   자체 계산 (외부 의존 제거)
    #   주문별 대표 시각 → 간격 → CV → 1/CV
    order_times = (
        normal
        .groupby(['CustomerID', 'InvoiceNo'])['InvoiceDate']
        .max()
        .reset_index()
        .sort_values(['CustomerID', 'InvoiceDate'])
    )

    order_times['prev_date'] = (
        order_times.groupby('CustomerID')['InvoiceDate'].shift(1)
    )
    order_times['interval'] = (
        order_times['InvoiceDate'] - order_times['prev_date']
    ).dt.days

    interval_stats = (
        order_times
        .groupby('CustomerID')['interval']
        .agg(['mean', 'std'])
    )

    interval_cv = interval_stats['std'] / interval_stats['mean']

    # 1/CV: 높을수록 리듬 안정적
    # CV=0 (간격 완전 동일) → 1/0 = inf → NaN 처리
    # CV=NaN (1~2회 구매) → NaN 유지
    rhythm_score = (1 / interval_cv).replace([np.inf, -np.inf], np.nan)
    rhythm_score.name = 'Purchase_Rhythm_Score'

    # ── Step 6: 결합 ─────────────────────────────────────────────
    consistency_features = pd.concat(
        [cat_std, rev_cv, rhythm_score],
        axis=1
    )

    consistency_features = consistency_features[[
        'Category_Concentration_Std', 'Revenue_Consistency',
        'Purchase_Rhythm_Score'
    ]]
    consistency_features.index.name = 'CustomerID'
    consistency_features = consistency_features.reset_index()

    # 결측 참고
    n_nan_cat = consistency_features['Category_Concentration_Std'].isna().sum()
    n_nan_rev = consistency_features['Revenue_Consistency'].isna().sum()
    n_nan_rhythm = consistency_features['Purchase_Rhythm_Score'].isna().sum()

    # ── Step 7: 넛지 유형 매핑 메타데이터 ────────────────────────
    nudge_mapping = {
        'Category_Concentration_Std': {
            '넛지_유형': '일관성(consistency)',
            '이론적_근거': '인지부조화 이론 — 낮을수록 시간이 지나도 '
                         '동일 카테고리를 유지하는 일관된 취향, '
                         '부조화를 회피하는 행동 지표 '
                         '(논문 제안서 핵심 변수)',
            '활용_목적': '낮을수록 일관된 취향으로 분류',
            '선행연구_대응': 'Yang Liu(2025)에 없는 변수, 본 연구 신규',
            '결측_전략': '1개월만 구매한 고객 → NaN 유지'
        },
        'Revenue_Consistency': {
            '넛지_유형': '일관성(consistency)',
            '이론적_근거': '단계적 순응 — 낮을수록 매번 비슷한 금액을 지출, '
                         '"이전과 같은 수준" 유지 성향이 강함, '
                         '높을수록 지출 패턴의 인지부조화 발생',
            '활용_목적': '지출 일관성 측정',
            '선행연구_대응': 'Yang Liu(2025) spending volatility와 동일 개념, '
                          '인지부조화 해석 추가',
            '결측_전략': '1회 구매자 → NaN 유지'
        },
        'Purchase_Rhythm_Score': {
            '넛지_유형': '일관성(consistency) / 타성(inertia)',
            '이론적_근거': '단순측정 효과 — 높을수록 일정한 주기로 구매, '
                         '"다음에도 같은 시기에 사겠다"는 자기 일관성 유지, '
                         '동시에 구매 리듬이라는 디폴트 패턴(타성)과도 연결',
            '활용_목적': '구매 리듬 안정성 측정 (카테고리 2 CV_Interval의 역수)',
            '선행연구_대응': 'Yang Liu(2025)에 없는 변수, 본 연구 신규',
            '결측_전략': '1~2회 구매자 또는 간격 완전 동일 → NaN 유지'
        }
    }

    # ── Step 8: 검증 출력 ────────────────────────────────────────
    if verbose:
        print(f"\n  생성된 고객 수: {len(consistency_features):,}명")
        print(f"  Category_Concentration_Std NaN: {n_nan_cat:,}명")
        print(f"  Revenue_Consistency NaN: {n_nan_rev:,}명")
        print(f"  Purchase_Rhythm_Score NaN: {n_nan_rhythm:,}명")
        print(f"  변수 목록: {list(consistency_features.columns[1:])}")
        print("-" * 65)
        print("  [넛지 유형 매핑 — 강준만(2016) 일관성(consistency)]")
        for var, info in nudge_mapping.items():
            print(f"    {var:.<30s} {info['넛지_유형']}")
        print("-" * 65)
        print("  [기술 통계]")
        print(consistency_features.describe().round(4).to_string())
        print("=" * 65)

    return consistency_features, nudge_mapping

In [22]:
consistency_feat, nudge_map8 = build_category8_consistency(df_final)
features = features.merge(consistency_feat, on='CustomerID', how='left')

카테고리 8: 일관성 변수 (일관성 넛지) 생성 시작
  정상 주문 필터링: 526,039행

  생성된 고객 수: 6,516명
  Category_Concentration_Std NaN: 3,812명
  Revenue_Consistency NaN: 3,671명
  Purchase_Rhythm_Score NaN: 4,517명
  변수 목록: ['Category_Concentration_Std', 'Revenue_Consistency', 'Purchase_Rhythm_Score']
-----------------------------------------------------------------
  [넛지 유형 매핑 — 강준만(2016) 일관성(consistency)]
    Category_Concentration_Std.... 일관성(consistency)
    Revenue_Consistency........... 일관성(consistency)
    Purchase_Rhythm_Score......... 일관성(consistency) / 타성(inertia)
-----------------------------------------------------------------
  [기술 통계]
         CustomerID  Category_Concentration_Std  Revenue_Consistency  Purchase_Rhythm_Score
count  6.516000e+03                   2704.0000            2845.0000              1999.0000
mean   4.969846e+05                      0.0100               0.4910                 2.4162
std    6.801235e+05                      0.0028               0.3132                 8.1111
min    

# 종속변수 생성

In [23]:
features

,CustomerID,Recency,Frequency,Monetary,AOV,Avg_Items_per_Order,Active_Months,Purchase_Span,Hour_Entropy,DayOfWeek_Entropy,...,Has_Return,Days_to_Return_Purchase,Cancellation_Rate,Return_Recovery_Ratio,Country_Trend_Alignment,Country_Purchase_Rank,UK_Flag,Category_Concentration_Std,Revenue_Consistency,Purchase_Rhythm_Score
0,12346,325,1,77183.60,77183.600000,74215.000000,1,0,0.000000,0.000000,...,1,NaN,0.5,0,0.000000,0.999338,1,NaN,NaN,NaN
1,12347,2,7,4310.96,615.851429,351.142857,7,365,0.514918,0.709319,...,0,NaN,0.0,0,0.097087,1.000000,0,0.011564,0.553836,3.265000
2,12348,75,4,1599.24,399.810000,585.250000,4,282,0.290463,0.474383,...,0,NaN,0.0,0,0.181818,0.750000,0,0.011574,0.618758,1.339989
3,12349,18,1,1453.60,1453.600000,631.000000,1,0,0.000000,0.000000,...,0,NaN,0.0,0,0.150685,0.714286,0,NaN,NaN,NaN
4,12350,310,1,312.40,312.400000,197.000000,1,0,0.000000,0.000000,...,0,NaN,0.0,0,0.176471,0.200000,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6511,1481435,1,1,3.35,3.350000,2.000000,1,0,0.000000,0.000000,...,0,NaN,0.0,0,0.000000,0.080010,1,NaN,NaN,NaN
6512,1481439,1,1,5661.52,5661.520000,1748.000000,1,0,0.000000,0.000000,...,0,NaN,0.0,0,0.338583,0.964753,1,NaN,NaN,NaN
6513,1481492,0,1,6725.54,6725.540000,2011.000000,1,0,0.000000,0.000000,...,0,NaN,0.0,0,0.283174,0.973854,1,NaN,NaN,NaN
6514,1481497,0,1,3161.52,3161.520000,654.000000,1,0,0.000000,0.000000,...,0,NaN,0.0,0,0.732143,0.911137,1,NaN,NaN,NaN


In [24]:
def build_dependent_variables(
    features,
    df_raw,
    churn_thresholds=(30, 60, 90),
    default_churn_threshold=90,
    verbose=True,
):
    """
    고객 단위 파생변수에 종속변수를 추가한다.

    Churn은 Recency 기준을 30/60/90일로 각각 라벨링해 A/B 데이터셋에서
    이탈 샘플 수가 어떻게 달라지는지 비교할 수 있게 한다.
    """
    result = features.copy()
    df = df_raw.copy()
    df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
    df['Revenue'] = df['Quantity'] * df['UnitPrice']
    df['is_cancel'] = df['InvoiceNo'].astype(str).str.startswith('C').astype(int)

    if verbose:
        print("=" * 60)
        print("종속변수 생성")

    # ── Main: Churn thresholds ───────────────────────────────────
    churn_targets = []
    for threshold in churn_thresholds:
        target_col = f'Churn_{threshold}'
        result[target_col] = (result['Recency'] > threshold).astype(int)
        churn_targets.append(target_col)

        if verbose:
            r = result[target_col].mean()
            print(f"  [Main] {target_col} (Recency > {threshold}일)")
            print(f"    이탈: {result[target_col].sum():,}명 ({r:.1%})")
            print(f"    잔존: {(result[target_col] == 0).sum():,}명 ({1-r:.1%})")

    default_col = f'Churn_{default_churn_threshold}'
    if default_col not in result.columns:
        raise ValueError(f"default_churn_threshold={default_churn_threshold} is not in {churn_thresholds}")
    result['Churn'] = result[default_col]

    # ── Sub: ImpulsePurchaseIndex ─────────────────────────────────
    normal = df[df['is_cancel'] == 0].copy()
    target_cids = set(result['CustomerID'])
    normal = normal[normal['CustomerID'].isin(target_cids)]

    # 주문별 매출
    orev = normal.groupby(['CustomerID', 'InvoiceNo'])['Revenue'].sum().reset_index(name='orev')
    aov = orev.groupby('CustomerID')['orev'].mean().reset_index(name='aov')
    orev = orev.merge(aov, on='CustomerID')

    # 주문별 신규 카테고리 포함 여부
    normal_s = normal.sort_values(['CustomerID', 'InvoiceDate'])
    seen = {}
    new_flags = {}
    for cid, inv, sc in zip(normal_s['CustomerID'], normal_s['InvoiceNo'], normal_s['StockCode']):
        if cid not in seen:
            seen[cid] = set()
        k = (cid, inv)
        if k not in new_flags:
            new_flags[k] = False
        if sc not in seen[cid]:
            new_flags[k] = True
        seen[cid].add(sc)

    nf = pd.DataFrame([
        {'CustomerID': k[0], 'InvoiceNo': k[1], 'has_new': v}
        for k, v in new_flags.items()
    ])
    orev = orev.merge(nf, on=['CustomerID', 'InvoiceNo'], how='left')
    orev['is_impulse'] = (
        (orev['orev'] > orev['aov'] * 1.5) & orev['has_new']
    ).astype(int)

    imp = orev.groupby('CustomerID').agg(
        ImpulsePurchaseIndex=('is_impulse', 'mean'),
        ImpulseFlag=('is_impulse', 'max'),
    ).reset_index()

    result = result.merge(imp, on='CustomerID', how='left')
    result['ImpulsePurchaseIndex'] = result['ImpulsePurchaseIndex'].fillna(0)
    result['ImpulseFlag'] = result['ImpulseFlag'].fillna(0).astype(int)

    if verbose:
        ir = result['ImpulseFlag'].mean()
        print(f"  [Sub]  Impulse (AOV×1.5 초과 & 신규카테고리)")
        print(f"    충동 경험: {result['ImpulseFlag'].sum():,}명 ({ir:.1%})")
        print(f"    비충동:    {(result['ImpulseFlag'] == 0).sum():,}명 ({1-ir:.1%})")
        print(f"    평균 비율: {result['ImpulsePurchaseIndex'].mean():.3f}")
        print("=" * 60)

    return result


In [25]:
features = build_dependent_variables(features, df_final)

종속변수 생성
  [Main] Churn_30 (Recency > 30일)
    이탈: 4,617명 (70.9%)
    잔존: 1,899명 (29.1%)
  [Main] Churn_60 (Recency > 60일)
    이탈: 3,671명 (56.3%)
    잔존: 2,845명 (43.7%)
  [Main] Churn_90 (Recency > 90일)
    이탈: 3,044명 (46.7%)
    잔존: 3,472명 (53.3%)
  [Sub]  Impulse (AOV×1.5 초과 & 신규카테고리)
    충동 경험: 1,370명 (21.0%)
    비충동:    5,146명 (79.0%)
    평균 비율: 0.053


In [26]:
features

,CustomerID,Recency,Frequency,Monetary,AOV,Avg_Items_per_Order,Active_Months,Purchase_Span,Hour_Entropy,DayOfWeek_Entropy,...,UK_Flag,Category_Concentration_Std,Revenue_Consistency,Purchase_Rhythm_Score,Churn_30,Churn_60,Churn_90,Churn,ImpulsePurchaseIndex,ImpulseFlag
0,12346,325,1,77183.60,77183.600000,74215.000000,1,0,0.000000,0.000000,...,1,NaN,NaN,NaN,1,1,1,1,0.000000,0
1,12347,2,7,4310.96,615.851429,351.142857,7,365,0.514918,0.709319,...,0,0.011564,0.553836,3.265000,0,0,0,0,0.142857,1
2,12348,75,4,1599.24,399.810000,585.250000,4,282,0.290463,0.474383,...,0,0.011574,0.618758,1.339989,1,1,0,0,0.250000,1
3,12349,18,1,1453.60,1453.600000,631.000000,1,0,0.000000,0.000000,...,0,NaN,NaN,NaN,0,0,0,0,0.000000,0
4,12350,310,1,312.40,312.400000,197.000000,1,0,0.000000,0.000000,...,0,NaN,NaN,NaN,1,1,1,1,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6511,1481435,1,1,3.35,3.350000,2.000000,1,0,0.000000,0.000000,...,1,NaN,NaN,NaN,0,0,0,0,0.000000,0
6512,1481439,1,1,5661.52,5661.520000,1748.000000,1,0,0.000000,0.000000,...,1,NaN,NaN,NaN,0,0,0,0,0.000000,0
6513,1481492,0,1,6725.54,6725.540000,2011.000000,1,0,0.000000,0.000000,...,1,NaN,NaN,NaN,0,0,0,0,0.000000,0
6514,1481497,0,1,3161.52,3161.520000,654.000000,1,0,0.000000,0.000000,...,1,NaN,NaN,NaN,0,0,0,0,0.000000,0


# 전처리 최종 마무리

In [27]:
# ── 상수 정의 ────────────────────────────────────────────────────────────

CHURN_THRESHOLDS = [30, 60, 90]
CHURN_TARGETS = [f'Churn_{d}' for d in CHURN_THRESHOLDS]
ID_TARGET_COLS = [
    'CustomerID', 'Churn', *CHURN_TARGETS,
    'ImpulsePurchaseIndex', 'ImpulseFlag',
]

DROP_MULTICOLLINEAR = [
    'Revenue_per_Visit',        # AOV와 r=1.000
    'Avg_Basket_Size',          # Avg_Items_per_Order와 r=1.000
    'Revenue_Trend',            # AOV와 r=0.981
    'Month_Entropy',            # Hour/DayOfWeek/Active_Months와 0.92~0.94
    'Unique_Products_Ratio',    # Repurchase_Rate와 r=-0.963
    'Single_Item_Order_Ratio',  # Top_Product_Concentration과 r=0.942
    'Has_Return',               # Cancellation_Rate과 r=0.871
    'High_Price_Item_Ratio',    # Avg_Unit_Price와 r=0.869
]

WINSORIZE_COLS = [
    'Monetary', 'AOV', 'Avg_Items_per_Order',
    'Max_Single_Order_Revenue', 'Avg_Unique_Items_per_Basket', 'Frequency',
]

TIMESERIES_COLS = [
    'IPI_Mean', 'IPI_Std', 'CV_Interval',
    'Revenue_Consistency', 'Purchase_Rhythm_Score',
    'Category_Concentration_Std',
]

FEATURE_GROUPS = {
    'RFM_baseline': [
        'Recency', 'Frequency', 'Monetary', 'AOV',
        'Avg_Items_per_Order', 'Active_Months', 'Purchase_Span',
        'Revenue_Consistency',
    ],
    '인지적효율성': [
        'Basket_Diversity_Index', 'Avg_Unique_Items_per_Basket',
        'Large_Basket_Ratio', 'Max_Single_Order_Revenue',
    ],
    '유도성': [
        'Repurchase_Rate', 'Top_Product_Concentration',
        'Avg_Repurchase_Interval',
    ],
    '흥미성': [
        'New_Category_Ratio',
    ],
    '긍정성': [
        'Avg_Unit_Price', 'Price_Variance', 'Low_Price_Item_Ratio',
        'Return_Rate', 'Days_to_Return_Purchase',
        'Cancellation_Rate', 'Return_Recovery_Ratio',
    ],
    '비교성': [
        'Country_Trend_Alignment', 'Country_Purchase_Rank', 'UK_Flag',
    ],
    '일관성': [
        'Category_Concentration_Std',
    ],
    '타성': [
        'Hour_Entropy', 'DayOfWeek_Entropy', 'Preferred_Hour',
        'Weekend_Ratio', 'IPI_Mean', 'IPI_Std', 'CV_Interval',
        'Purchase_Rhythm_Score',
    ],
}


# ===========================================================================
# Step 1. 다중공선성 제거
# ===========================================================================
def remove_multicollinearity(df, drop_cols=DROP_MULTICOLLINEAR, verbose=True):
    """상관계수 |r| > 0.85 기반으로 중복 변수를 제거한다."""
    result = df.drop(columns=[c for c in drop_cols if c in df.columns])
    if verbose:
        print(f"[Step 1] 다중공선성 변수 {len(drop_cols)}개 제거 → {df.shape[1]} → {result.shape[1]} cols")
    return result


# ===========================================================================
# Step 2. 이상치 처리 + 왜도 변환
# ===========================================================================
def handle_outliers_and_skew(df, winsorize_cols=WINSORIZE_COLS,
                             skew_threshold=2.0, quantile_cap=0.99,
                             verbose=True):
    """
    (1) 지정 변수 상위 percentile 캡핑 (Winsorizing)
    (2) 왜도 > threshold이고 min >= 0인 변수에 log1p 변환
    (3) 무한대 → NaN 변환
    """
    result = df.copy()
    frequency_filter = result['Frequency'].copy() if 'Frequency' in result.columns else None
    feature_cols = [c for c in result.columns if c not in ID_TARGET_COLS]

    # ── Winsorizing ──
    if verbose:
        print(f"\n[Step 2-1] Winsorizing (상위 {quantile_cap*100:.0f}% 캡핑)")
    for col in winsorize_cols:
        if col in result.columns:
            upper = result[col].quantile(quantile_cap)
            n_clipped = (result[col] > upper).sum()
            result[col] = result[col].clip(upper=upper)
            if verbose:
                print(f"  {col}: 캡핑값={upper:.1f}, 영향={n_clipped}건")

    # ── Log1p 변환 ──
    log_transformed = []
    for col in feature_cols:
        if col in result.columns and result[col].dtype in ['float64', 'int64']:
            s = result[col].skew()
            if s > skew_threshold and result[col].min() >= 0:
                result[col] = np.log1p(result[col])
                log_transformed.append((col, round(s, 2)))

    if verbose:
        print(f"\n[Step 2-2] Log1p 변환 ({len(log_transformed)}개)")
        for col, s in log_transformed:
            print(f"  {col} (왜도: {s})")

    # ── 무한대 정리 ──
    result[feature_cols] = result[feature_cols].replace([np.inf, -np.inf], np.nan)

    # Frequency는 모델 입력에서는 변환값을 쓰되, A/B 샘플링 기준은 원본 구매 횟수를 유지한다.
    if frequency_filter is not None:
        result['_Frequency_Filter'] = frequency_filter
    return result


# ===========================================================================
# Step 3. Frequency=1 고객 처리 → A/B 실험 셋
# ===========================================================================
def _fill_remaining_na(df):
    """공통 결측치 처리."""
    result = df.copy()
    result['Days_to_Return_Purchase'] = result['Days_to_Return_Purchase'].fillna(0)
    result['Avg_Repurchase_Interval'] = result['Avg_Repurchase_Interval'].fillna(
        result['Avg_Repurchase_Interval'].median()
    )
    result['Price_Variance'] = result['Price_Variance'].fillna(0)
    result = result.fillna(0)
    return result


def _print_target_summary(result, label):
    """A/B 데이터셋별 종속변수 분포를 출력한다."""
    print(f"\n[{label}] Target 분포")
    print(f"  고객 수: {len(result):,}명")
    for target in CHURN_TARGETS:
        if target in result.columns:
            rate = result[target].mean()
            print(f"  {target}: {result[target].sum():,}명 ({rate:.1%})")
    if 'ImpulseFlag' in result.columns:
        print(f"  ImpulseFlag: {result['ImpulseFlag'].sum():,}명 ({result['ImpulseFlag'].mean():.1%})")
    print(f"  잔여 결측: {result.isnull().sum().sum()}")


def build_experiment_A(df, min_freq=2, verbose=True):
    """실험 A: Frequency ≥ min_freq 고객만 사용."""
    freq_col = '_Frequency_Filter' if '_Frequency_Filter' in df.columns else 'Frequency'
    result = df[df[freq_col] >= min_freq].copy()
    result = result.drop(columns=['_Frequency_Filter'], errors='ignore')

    for col in TIMESERIES_COLS:
        if col in result.columns:
            result[col] = result[col].fillna(result[col].median())

    result = _fill_remaining_na(result)

    if verbose:
        print(f"\n[Step 3-A] Frequency ≥ {min_freq}")
        _print_target_summary(result, "실험A")
    return result


def build_experiment_B(df, verbose=True):
    """실험 B: 전체 고객 유지 + has_timeseries flag 보존 + 중앙값 대체."""
    result = df.copy()
    freq_col = '_Frequency_Filter' if '_Frequency_Filter' in result.columns else 'Frequency'
    result['has_timeseries'] = (result[freq_col] >= 2).astype(int)
    result = result.drop(columns=['_Frequency_Filter'], errors='ignore')

    for col in TIMESERIES_COLS:
        if col in result.columns:
            result[col] = result[col].fillna(result[col].median())

    result = _fill_remaining_na(result)

    if verbose:
        print(f"\n[Step 3-B] 전체 고객 (flag 보존, 모델 feature에서는 제외)")
        _print_target_summary(result, "실험B")
    return result


# ===========================================================================
# Step 4. Feature Set 구성
# ===========================================================================
def get_feature_sets():
    """
    모델 실험용 Feature Set 2가지를 반환한다.

    Returns
    -------
    dict : {'rfm': 8개, 'rfm_hci': 35개}
    """
    rfm = FEATURE_GROUPS['RFM_baseline']
    hci = []
    for k, v in FEATURE_GROUPS.items():
        if k != 'RFM_baseline':
            hci.extend(v)
    rfm_hci = rfm + hci

    return {
        'rfm': rfm,
        'rfm_hci': rfm_hci,
    }


def print_feature_sets(feature_sets):
    """Feature Set 구성을 출력한다."""
    print("\n" + "=" * 60)
    print("[Step 4] Feature Set 구성")
    print("=" * 60)
    for name, cols in feature_sets.items():
        print(f"  {name}: {len(cols)}개")

    print("\n넛지 유형별 상세:")
    for group, vars in FEATURE_GROUPS.items():
        print(f"  [{group}] {vars}")


# ===========================================================================
# Step 5. 최종 검증
# ===========================================================================
def validate_dataset(df, feature_sets, name=""):
    """데이터셋 무결성을 검증한다."""
    print(f"\n--- 검증: {name} ---")
    print(f"  Shape: {df.shape}")
    print(f"  결측: {df.isnull().sum().sum()}")
    print(f"  Inf: {np.isinf(df.select_dtypes(include=[np.number])).sum().sum()}")

    missing = [f for f in feature_sets['rfm_hci'] if f not in df.columns]
    if missing:
        print(f"  ⚠️ 누락 변수: {missing}")
    else:
        print("  ✅ RFM+HCI 35개 변수 존재 확인")

    if len(feature_sets['rfm']) != 8 or len(feature_sets['rfm_hci']) != 35:
        print("  ⚠️ Feature 수 확인 필요")
    else:
        print("  ✅ Feature 수 확인: RFM 8개 / RFM+HCI 35개")


# ===========================================================================
# 실행
# ===========================================================================
if __name__ == '__main__':
    df = features
    print(f"원본: {df.shape}")

    df = remove_multicollinearity(df)
    df = handle_outliers_and_skew(df)

    df_A = build_experiment_A(df)
    df_B = build_experiment_B(df)

    feature_sets = get_feature_sets()
    print_feature_sets(feature_sets)

    validate_dataset(df_A, feature_sets, "실험A (Freq≥2)")
    validate_dataset(df_B, feature_sets, "실험B (전체)")


원본: (6516, 50)
[Step 1] 다중공선성 변수 8개 제거 → 50 → 42 cols

[Step 2-1] Winsorizing (상위 99% 캡핑)
  Monetary: 캡핑값=14339.4, 영향=66건
  AOV: 캡핑값=4482.5, 영향=66건
  Avg_Items_per_Order: 캡핑값=1638.0, 영향=66건
  Max_Single_Order_Revenue: 캡핑값=5348.0, 영향=66건
  Avg_Unique_Items_per_Basket: 캡핑값=417.7, 영향=66건
  Frequency: 캡핑값=25.0, 영향=63건

[Step 2-2] Log1p 변환 (14개)
  Frequency (왜도: 3.33)
  Monetary (왜도: 3.79)
  AOV (왜도: 3.69)
  Avg_Items_per_Order (왜도: 2.93)
  Active_Months (왜도: 2.34)
  Weekend_Ratio (왜도: 2.84)
  Avg_Unique_Items_per_Basket (왜도: 4.07)
  Max_Single_Order_Revenue (왜도: 3.42)
  Repurchase_Rate (왜도: 2.34)
  Avg_Unit_Price (왜도: 2.42)
  Return_Rate (왜도: 43.6)
  Days_to_Return_Purchase (왜도: 2.17)
  Cancellation_Rate (왜도: 2.04)
  Purchase_Rhythm_Score (왜도: 16.47)

[Step 3-A] Frequency ≥ 2

[실험A] Target 분포
  고객 수: 2,845명
  Churn_30: 1,398명 (49.1%)
  Churn_60: 881명 (31.0%)
  Churn_90: 602명 (21.2%)
  ImpulseFlag: 1,370명 (48.2%)
  잔여 결측: 0

[Step 3-B] 전체 고객 (flag 보존, 모델 feature에서는 제외)

[실험B] Target 분포
  고객

# 데이터셋 저장

In [28]:
# 저장
df_A.to_csv('modeling_ready_A_freqGE2.csv', index=False)
df_B.to_csv('modeling_ready_B_all.csv', index=False)

print("\n✅ 저장 완료: modeling_ready_A_freqGE2.csv / modeling_ready_B_all.csv")
print("   - Churn_30, Churn_60, Churn_90 라벨 포함")
print("   - 모델 feature set은 RFM 8개 vs RFM+HCI 35개로 구성")



✅ 저장 완료: modeling_ready_A_freqGE2.csv / modeling_ready_B_all.csv
   - Churn_30, Churn_60, Churn_90 라벨 포함
   - 모델 feature set은 RFM 8개 vs RFM+HCI 35개로 구성
